This notebook is the historical exploratory analysis from before the upgrade. Its forecasting and spatial results are superseded. Current results and verification limits are in [docs/report.md](../docs/report.md).

# California Grid Stress Monitoring Dashboard

**Analytical Notebook: EIA-930 California Balancing Authority Grid Operations | January–April 2026**

| Field | Details |
| --- | --- |
| Author | Sileshi Hirpa |
| Date | April-May 2026 |
| Data source | EIA Form EIA-930 - Balancing Authority Hourly Operations |
| Scope | California balancing authorities only: BANC, CISO, IID, LDWP, TIDC |
| Tools | Python, pandas, Plotly, Jupyter Notebook, Tableau Public |

---

## Executive Summary

This notebook presents a portfolio-ready analysis of public California electricity
grid operations data from EIA Form EIA-930. The analysis prepares hourly balancing
authority records, validates the California-only scope, converts timestamps to
Pacific Time, engineers a Stress Index, classifies review-priority hours, and
exports dashboard-ready datasets for Tableau Public.

The central analytical goal is to identify periods when electricity demand
approaches the observed peak level for each California balancing authority. Because
California balancing authorities operate at very different scales, the Stress Index
normalizes each authority against its own observed peak demand within the
January-April 2026 dataset window.

The notebook supports the public portfolio README by making the full analytical
workflow reproducible: raw data preparation, scope validation, feature engineering,
review-priority classification, key findings, limitations, and dashboard data model
export.

---

## Table of Contents

1. Business Question and Project Goals
2. Setup: Imports, Paths, and Constants
3. Data Loading and Initial Inspection
4. Data Cleaning and Timezone Conversion
5. Scope Validation and California Filtering
6. California Demand Visualization
7. Stress Index Feature Engineering
8. Review Priority Classification
9. Key Findings
10. Top Review Hours
11. Operational Context
12. Processed Output Export
13. Dashboard Data Model Export
14. Limitations and Caveats
15. Conclusion and Portfolio Summary

---

## Methodology

The notebook follows a reproducible analytics workflow:

1. **Data ingestion:** Load the public EIA-930 hourly balancing authority file
   using relative file paths.
2. **Data cleaning:** Standardize key column names, remove incomplete records,
   remove duplicate authority-hour records, convert UTC timestamps to Pacific Time,
   and coerce measurement fields to numeric values.
3. **Scope validation:** Inspect the balancing authorities present in the raw file,
   identify that the source contains records beyond California, and filter the
   working dataset to BANC, CISO, IID, LDWP, and TIDC.
4. **Visualization:** Build time-series views that compare California balancing
   authority demand patterns and reveal differences in scale and daily demand
   behavior.
5. **Feature engineering:** Compute `peak_demand_mw` for each balancing authority
   and calculate `stress_index = (demand_mw / peak_demand_mw) × 100`.
6. **Priority classification:** Use named threshold constants to classify each hour
   into High, Medium, or Low Review Priority.
7. **Evidence summary:** Compute key findings directly from the data, including
   priority-hour counts, peak demand, and average Stress Index.
8. **Dashboard export:** Create dashboard-ready detail and summary files for
   Tableau Public.
9. **Limitations:** Document the dataset window, public-data scope,
   non-historical peak denominator, and the fact that this is not an enterprise
   reliability model.

---

## Project Overview

This project uses publicly available EIA-930 data from the U.S. Energy Information
Administration and does not represent any utility's internal operational systems,
enterprise risk models, internal safety frameworks, or proprietary data pipelines.
The Stress Index denominator is derived from the January-April 2026 dataset window
only - not from a historical all-time record.

## Business Question

How can hourly electricity demand data be prepared, validated, and visualized so
that a reviewer can quickly identify periods that may deserve additional
operational attention?

## Project Goals

This notebook demonstrates:

1. Loading and inspecting publicly available hourly electricity demand data.
2. Cleaning and standardizing key fields for consistent downstream use.
3. Validating that the dataset scope matches the California-focused analysis objective.
4. Building time-series visualizations of demand patterns across balancing authorities.
5. Engineering a Stress Index based on demand relative to observed peak demand.
6. Classifying each hour into a review priority tier using the Stress Index.
7. Summarizing the highest-priority hours in a compact, decision-ready table.
8. Exporting processed and dashboard-ready datasets for use in Tableau and Power BI.

## Why This Analysis Matters

Grid reliability depends on understanding when demand approaches observed peak
levels. A reviewer who can quickly identify high-stress hours - and understand
whether local generation or imported energy was meeting that demand - is better
positioned to investigate potential reliability concerns. This notebook provides
that structure through systematic data preparation, feature engineering, and a
priority-ranked output table.

In [ ]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================
#%pip install plotly

import pandas as pd
import numpy as np
import plotly.express as px
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

## Step 1 Explanation: Importing Libraries

Before working with the dataset, I import the Python libraries I need for:

- reading the CSV file
- cleaning and transforming the data
- computing simple summary metrics
- creating interactive charts

This is a standard first step in a Jupyter notebook workflow.


In [ ]:
# ============================================
# STEP 2: DEFINE FILE PATHS AND CONSTANTS
# ============================================

DATA_PATH     = "../data/raw/EIA930_BALANCE_2026_Jan_Jun.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_DIR    = "../outputs"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Review priority classification thresholds
# Named constants make threshold values easy to locate and adjust
HIGH_REVIEW_THRESHOLD   = 90   # Stress Index >= 90  ->  High Review Priority
MEDIUM_REVIEW_THRESHOLD = 75   # Stress Index >= 75  ->  Medium Review Priority
TOP_N_REVIEW_HOURS      = 10   # Rows shown in the ranked review table

print("File paths configured.")
print("Output directories ready.")
print(f"Review thresholds: High >= {HIGH_REVIEW_THRESHOLD}, Medium >= {MEDIUM_REVIEW_THRESHOLD}")
print(f"Top review hours to display: {TOP_N_REVIEW_HOURS}")

## Step 2: File Paths and Configuration

This cell defines:

- the source data file path
- the output directories for processed data and summary tables
- named constants for review priority thresholds and display settings

Using named constants (`HIGH_REVIEW_THRESHOLD`, `MEDIUM_REVIEW_THRESHOLD`,
`TOP_N_REVIEW_HOURS`) makes the classification logic readable and the threshold
values easy to find and adjust without editing the function body directly. They
are defined here - in the configuration section - so any reader encounters them
before the classification step.

The `os.makedirs(exist_ok=True)` calls ensure output directories exist at
runtime, preventing export errors if they have not been created manually.

### Step 3: Load and Inspect Raw Dataset

The first technical step in the analysis is to load the raw EIA-930 balancing authority dataset into a pandas DataFrame. At this stage, the dataset is intentionally kept in its original structure. No filtering, aggregation, renaming, or transformation is performed before the initial inspection.

This step serves four purposes. First, it confirms that the file path is correct and that the dataset can be read successfully. Second, it verifies the number of rows and columns in the source file. Third, it displays the original column names so that the structure of the raw data is documented before any cleaning takes place. Finally, it helps identify which fields are most relevant for the California grid analysis, including balancing authority, date and time fields, electricity demand, demand forecast, net generation, and total interchange.

Inspecting the raw dataset before applying transformations is an important data quality practice. It helps detect structural issues early, such as unexpected column names, inconsistent formatting, missing fields, or columns that may require standardization before analysis. This creates a clear audit trail from the original source file to the cleaned analytical dataset used later in the notebook.

### Step 3B: Standardize Known Raw Column-Name Inconsistencies

After inspecting the raw column names, two minor naming inconsistencies were identified in the source dataset. One column contained an extra space before the adjusted label, and another column appeared to contain a spelling inconsistency in the phrase describing solar generation with integrated battery storage.

These issues do not affect the underlying energy values, but they can create problems later in the notebook if column names are referenced directly in code. For example, small differences in spacing or spelling can cause selection, renaming, aggregation, or dashboard export steps to fail even when the intended data column is present.

To prevent these issues, the affected column names are standardized before the main analysis columns are renamed. This keeps the raw data values unchanged while making the dataset easier to work with programmatically. The correction is limited to column labels only and does not modify any measured electricity demand, generation, or interchange values.

This step improves reproducibility, reduces the risk of downstream errors, and ensures that any dashboard-ready outputs use clean and consistent field names.

In [ ]:
from IPython.display import display, Markdown

# ============================================
# STEP 3: LOAD AND PREVIEW DATA
# ============================================

# 1. Load the dataset
df = pd.read_csv(DATA_PATH, low_memory=False)

# 2. Build and render the Markdown schema summary directory
md_table = f"### 📊 Grid Analytics Dataset Schema Summary\n"
md_table += f"**Total Records:** {df.shape[0]:,} rows | **Total Metrics:** {df.shape[1]} columns\n\n"
md_table += "| # | Registered Metric Field |\n|---|---|\n"

for idx, col in enumerate(df.columns, 1):
    md_table += f"| **{idx}** | `{col}` |\n"

display(Markdown(md_table))

print("\n" + "=" * 50)
print("🔬 EXECUTIVE TRANSPOSED PREVIEW (First 3 Rows):")
print("=" * 50)

# 3. Render the vertical transposed row preview
display(df.head(3).T)

In [ ]:
# Preview actual data for the core analysis columns using their original raw names
core_cols_raw = [
    "Balancing Authority",
    "Data Date",
    "Hour Number",
    "Demand (MW)",
    "Net Generation (MW)",
    "Total Interchange (MW)",
]
display(df[core_cols_raw].head())

### Ingestion Insights & Strategic Pipeline Alignment

#### 1. The Raw Baseline Paradox
Our initial data pull in **Step 3** successfully ingested a massive nationwide footprint of **158,182 hourly records across 65 columns**. However, a structural audit of the raw dataset immediately reveals a major geographic mismatch: the top of the file contains telemetry from the **AECI** (Associated Electric Cooperative, Inc.) network operating inside the **MIDW** (Midwest) balancing region. 

Because this federal repository aggregates concurrent electrical load metrics from dozens of multi-state networks, it serves as an immediate data-quality warning: **running predictive models or descriptive statistics on the unfiltered national baseline will heavily corrupt our California-specific insights.**

#### 2. The Physical Law of Grid Balance
Inspecting our targeted core columns preview yields a fundamental validation of engineering identity. In grid physics, the net flow of electricity must balance perfectly to the decimal point at any given hour following the identity:

$$\text{Net Generation (MW)} - \text{Demand (MW)} = \text{Total Interchange (MW)}$$

Looking at the very first operational hour (**Row 0**):
* **Demand:** $2,625.00 \text{ MW}$ (The active consumer load drawing from the network).
* **Net Generation:** $2,942.00 \text{ MW}$ (The power actively supplied by local generation facilities).
* **Total Interchange:** $+317.00 \text{ MW}$ (A positive value indicating an active surplus).

Because local generation exceeds immediate internal demand, the balancing authority is physically exporting its excess $+317 \text{ MW}$ to neighboring balancing authorities to prevent system over-frequency. This provides an excellent point of comparison for our upcoming scope, as California's primary network (**CISO**) historically operates with a heavily *negative* interchange—signaling a constant structural reliance on out-of-state power imports to protect grid reliability.

---

### 🚀 Critical Path: Why the Pipeline Must Move Forward

While **Step 4** cleanly standardizes our column labels into developer-friendly `snake_case`, the project cannot stop there. We must advance immediately to **Step 5 (Basic Data Cleaning & Timezone Conversion)** to resolve three severe engineering vulnerabilities present in our raw data:

* **Eliminating Multi-Million MW Contamination:** A deeper statistical check reveals that out-of-state entities (such as `WACM`) contain severe data-entry anomalies with erratic demand spikes exceeding **1,000,000 MW to 2,500,000 MW**. Step 5 introduces strict subset row cleanses to insulate our core dataframe before these massive non-California outliers can bias our downstream models or skew our visualization axes.
* **Standardizing the Human Timeline (UTC to PDT):** Electricity demand patterns are directly coupled to localized human behavior (afternoon cooling demands, morning industrial ramps). The raw data records timestamps natively in Universal Coordinated Time (`UTC`). If left unconverted, an extreme evening peak in Los Angeles at 7:00 PM PDT would register incorrectly as 2:00 AM UTC the next day. Step 5 enforces a localized Pacific Time conversion (`America/Los_Angeles`), grounding our machine learning inputs in geographic reality.
* **Strict Numerical Type Coercion:** Flat CSV parsers frequently read blank fields or text placeholders as generic Python objects (`dtype: object`). Step 5 forces our target electricity indicators into high-precision `float64` format, ensuring that subsequent cross-correlation arrays, feature scaling, and visualization layers execute flawlessly without raising type errors.

In [ ]:
# Standardize known raw EIA-930 column-name inconsistencies
df = df.rename(
    columns={
        "Net Generation (MW) from Pumped Storage  (Adjusted)": "Net Generation (MW) from Pumped Storage (Adjusted)",
        "Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted)": "Net Generation (MW) from Solar with Integrated Battery Storage (Adjusted)",
    }
)

In [ ]:
# Confirm the corrected columns exist
columns_to_check = [
    "Net Generation (MW) from Pumped Storage (Adjusted)",
    "Net Generation (MW) from Solar with Integrated Battery Storage (Adjusted)"
]

for col in columns_to_check:
    print(col, ":", col in df.columns)

**Column-name standardization only:**  
This cell corrects minor spacing and spelling inconsistencies in selected raw column labels. It does not alter the numerical data values.

### Step 4: Rename Important Columns for Analysis

The raw EIA-930 dataset contains long column names with spaces, parentheses, and measurement labels. These names are useful for understanding the source file, but they are not ideal for repeated analysis in Python. To make the notebook easier to read and less error-prone, the main fields used in this project are renamed using shorter, consistent snake_case names.

This step focuses only on the core columns needed for the California grid analysis, including balancing authority, date and time fields, electricity demand, demand forecast, net generation, and total interchange. The remaining source columns are left unchanged unless they are needed later.

Renaming columns at this stage improves code readability and creates a cleaner foundation for later filtering, aggregation, visualization, and dashboard export. The operation changes column labels only. It does not change the underlying data values.

In [ ]:
# ============================================
# STEP 4: RENAME IMPORTANT COLUMNS FOR ANALYSIS
# ============================================

# Mapping dictionary for standardizing core telemetry metrics into snake_case
rename_dict = {
    "Balancing Authority": "balancing_authority",
    "Data Date": "data_date",
    "Hour Number": "hour_number",
    "UTC Time at End of Hour": "utc_time_at_end_of_hour",
    "Demand Forecast (MW)": "demand_forecast_mw",
    "Demand (MW)": "demand_mw",
    "Net Generation (MW)": "net_generation_mw",
    "Total Interchange (MW)": "total_interchange_mw",
}

# Apply column renaming explicitly (avoiding inplace=True for better optimization)
df = df.rename(columns=rename_dict)

print("✅ Columns renamed successfully.")
print(f"📋 Current Dataset Footprint: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n🔍 Active Alignment Map:")
print("-" * 65)
for old_col, new_col in rename_dict.items():
    print(f"  ▪️ {old_col:<30} ➔  {new_col}")
print("-" * 65)

print("\n⭐ First 5 Standardized Features:")
print(df.columns.tolist()[:5])

In [ ]:
required_columns = [
    "balancing_authority",
    "data_date",
    "hour_number",
    "utc_time_at_end_of_hour",
    "demand_forecast_mw",
    "demand_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if len(missing_columns) == 0:
    print("All required analysis columns are present.")
else:
    print("Missing required columns:", missing_columns)

## Step 4 Summary

The key source columns were successfully renamed into shorter, consistent, and developer-friendly `snake_case` names. The dataset shape remained unchanged at **158,182 rows and 65 columns**, confirming that the operation modified structural column labels only and did not remove or alter any underlying records.

To maximize performance optimization and align with modern Pandas best practices, the mapping dictionary was applied using explicit variable reassignment (`df = df.rename(...)`) rather than using the discouraged `inplace=True` parameter. An automated visual alignment map was generated to explicitly audit the transformation from raw headers (such as `Balancing Authority` and `Demand (MW)`) to their clean programmatic counterparts (`balancing_authority`, `demand_mw`).

The validation check confirmed that all required analysis columns are present with a missing error length of zero. With the core data schema standardized and verified, the pipeline can safely proceed to the next stage of data type conversion, structural cleaning, and timezone adjustments in Step 5.

### Step 5: Basic Data Cleaning & Temporal Alignment

With the core analysis schema standardized into `snake_case`, the dataset enters the primary programmatic cleaning phase. This stage applies defensive data engineering constraints to guarantee structural integrity before executing any regional filtering or visualization layers. 

The cleaning pipeline systematically addresses four data-quality vulnerabilities:
* **Enforcing Completeness (Row Cleansing):** Drops records missing essential telemetry fields (`balancing_authority`, `utc_time_at_end_of_hour`, `demand_mw`) to eliminate incomplete observations. This process, alongside duplicate mitigation, safely contracts our tracking footprint from **158,182 down to 137,194 high-integrity rows**.
* **Deduplication:** Isolates and drops duplicate concurrent hourly records for individual balancing authorities, ensuring that every authority-hour intersection contains a unique, single telemetry point.
* **Temporal Standardization & Pacific Mapping:** Converts raw date and timestamp strings into timezone-aware `datetime64[ns, UTC]` objects. Crucially, because this project evaluates the California grid, a localized `local_time_pacific` column (`America/Los_Angeles`) is programmatically engineered from the UTC anchor. This grounds all downstream feature engineering in true geographic reality—ensuring peak evening loads are evaluated exactly when they physically hit the Western Interconnection rather than mapping to the next calendar day in UTC.
* **Strict Type Coercion:** Flat CSV parsers naturally read missing or corrupted entries as generic text objects (`dtype: object`). This stage loops through our core electricity indicators (`demand_mw`, `demand_forecast_mw`, `net_generation_mw`, `total_interchange_mw`) and forces them into high-precision floating-point numbers (`float64`), enabling flawless downstream statistical modeling and visualization execution.

*Note: Data transformation is intentionally restricted at this boundary to retain the raw granular structure of the data. Aggregations, scaling factors, and regional subset allocations are deferred to the upcoming Exploratory Data Analysis (EDA) and Scope Validation gates.*

In [ ]:
# ============================================
# STEP 5: BASIC DATA CLEANING & TEMPORAL MAPPING
# ============================================

# Record initial row count for data lineage auditing
initial_rows = len(df)

# 1. Enforce completeness by removing records missing essential telemetry fields
df = df.dropna(subset=["balancing_authority", "utc_time_at_end_of_hour", "demand_mw"])

# 2. Mitigate concurrency anomalies by dropping duplicate hourly authority records
df = df.drop_duplicates(subset=["balancing_authority", "utc_time_at_end_of_hour"])

# 3. Standardize date and timestamp strings into temporal objects
df["data_date"] = pd.to_datetime(df["data_date"], errors="coerce")
df["utc_time_at_end_of_hour"] = pd.to_datetime(
    df["utc_time_at_end_of_hour"], utc=True, errors="coerce"
)

# 4. Programmatically map the timeline to Pacific Time for localized grid context
df["local_time_pacific"] = df["utc_time_at_end_of_hour"].dt.tz_convert(
    "America/Los_Angeles"
)

# 5. Enforce strict numerical type coercion across core electricity metrics
numeric_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate data lineage shrinkage
final_rows = df.shape[0]
rows_removed = initial_rows - final_rows

print("✅ Basic data cleaning and temporal alignment complete.")
print("-" * 65)
print(f"📋 Initial Row Count:  {initial_rows:,}")
print(f"❌ Records Filtered:   {rows_removed:,} (Incomplete / Duplicates)")
print(f"🚀 High-Integrity Rows: {final_rows:,} × {df.shape[1]} columns")
print("-" * 65)

print("\n⚙️ Core Feature Datatype Schema Verification:")
print("=" * 65)
target_features = [
    "balancing_authority",
    "data_date",
    "utc_time_at_end_of_hour",
    "local_time_pacific",
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]

# Print a beautiful scannable layout of your dtypes
for feature, dtype in df[target_features].dtypes.items():
    print(f"  ▪️ {feature:<25} ➔  {str(dtype)}")
print("=" * 65)

## Step 5 Summary

The basic data cleaning and temporal alignment phase successfully compressed the tracking footprint from an initial **158,182 rows down to 137,194 high-integrity rows**, while expanding the schema to **66 columns**. 

The **13% reduction in row volume (20,988 records filtered)** represents a critical data-lineage milestone. This contraction was driven entirely by programmatic constraints designed to enforce structural completeness: eliminating incomplete telemetry logs missing actual `demand_mw` entries and purging duplicate, concurrent hourly reporting artifacts across individual balancing authorities.

Key systemic enhancements implemented during this gate include:
* **Temporal Mapping:** Core string timestamp attributes were successfully cast into timezone-aware `datetime64[ns, UTC]` objects. From this stable anchor, a localized `local_time_pacific` (`America/Los_Angeles`) column was engineered. Because California's electrical demand profiles are fundamentally coupled with local behavioral shifts—such as evening cooking peaks and solar generation drop-offs—this localized timeline ensures downstream feature engineering remains bound to physical regional reality.
* **Deterministic Type Coercion:** To prevent runtime exceptions during statistical analysis or dashboard visualization rendering, explicit type casting was enforced across our four primary operational metrics:
  * `demand_mw` (`float64`)
  * `demand_forecast_mw` (`float64`)
  * `net_generation_mw` (`float64`)
  * `total_interchange_mw` (`float64`)

The structural schema is now certified clean and mathematically normalized. The pipeline is fully prepared to advance to **Step 6: Review the Full Dataset**, where we will evaluate overall statistical distributions and conduct a targeted audit of extreme national-scale outliers.

### Step 6: Full Dataset Statistical Audit & Outlier Detection

Following the basic cleaning phase, the complete national dataset undergoes a comprehensive structural and statistical audit. This phase serves as an essential data validation gate—verifying that our data types have successfully transformed across all 66 columns and establishing an empirical baseline before the pipeline branches into California-specific regional filtering and advanced feature engineering.

The diagnostic review evaluates the data matrix using two primary vectors:
* **Structural Validation (`info()`):** Conducts a full memory and data-type audit across all 66 active columns. This trace explicitly verifies that our temporal mapping and programmatic string-to-numeric type coercions from Step 5 successfully applied without leaving trailing string objects (`dtype: object`) in our core telemetry channels.
* **Empirical Limits Audit (`describe()`):** Computes descriptive statistical aggregates (mean, standard deviation, percentiles, and absolute boundary minima/maxima) across our primary electricity metrics. Rather than looking at rows in isolation, this matrix provides a global view of the data's true scale—acting as a critical diagnostic layer to expose extreme, multi-million megawatt data-entry anomalies (such as reporting errors from out-of-state balancing authorities like `WACM`) that would heavily skew downstream visualizations and predictive baselines if left unmitigated.

In [ ]:
# ============================================
# STEP 6 A: FULL DATASET STATISTICAL AUDIT
# ============================================

print("⚙️  DF SCHEMA INTEGRITY SNAPSHOT")
print("=" * 65)
# Run a clean info trace to verify type transformations and memory boundaries
df.info()
print("=" * 65)

print("\n📊 EMPIRICAL BOUNDARY & DISTRIBUTION AUDIT")
print("-" * 65)

core_numeric_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]

# Generate and display the high-precision descriptive statistics table
stats_df = df[core_numeric_cols].describe()
display(stats_df)
print("-" * 65)

## Step 6 A Summary

The full dataset statistical audit confirmed that the cleaned dataframe contains **137,194 rows and 66 columns**, occupying a stable **70.1 MB** of memory. The diagnostic check verifies that all temporal anchors have successfully transformed—including the newly engineered, timezone-aware `local_time_pacific` field—and that our primary grid tracking metrics have successfully coerced to high-precision `float64` numeric values. 

However, this global profiling confirms that **the data does not yet isolate the five California balancing authorities** and exposes severe, systemic data-quality issues across the broader national footprint that must be addressed before visualization or dashboard ingestion:

* **Extreme Multi-Million MW Anomalies:** The empirical distribution table reveals a catastrophic maximum peak for both `demand_mw` (**2,576,875.00 MW**) and `net_generation_mw` (**2,576,906.00 MW**). Because the entire combined United States electrical infrastructure operates at a peak well below 800,000 MW, these values represent extreme out-of-state data-entry errors (primarily from non-California entities like the `WACM` balancing authority).
* **Telemetry Transgression (Negative Load Bounds):** The absolute minimum boundaries show severe degradation, with actual demand plummeting to **-40,972.00 MW**. Because standard customer load profiles cannot physically drop below zero, these records indicate unmitigated placeholder entries or corrupted reporting logs within the federal repository.
* **Minor Completeness Flaws:** While core tracking fields are completely populated, secondary indicators like `demand_forecast_mw` (136,641 non-null) and `total_interchange_mw` (137,047 non-null) exhibit minor data gaps, though their footprint remains negligible relative to the full matrix population.

### ➔ Strategic Verdict
This validation gate delivers the absolute engineering justification for our upcoming notebook workflow. It proves that running a generic or universal data model over the raw national footprint will severely skew our statistical baselines and warp dashboard visualization axes. To preserve analytic integrity, we must immediately proceed to **Step 6B & 6C** to trace these specific anomalies, and then execute our strict geographical filter to isolate the true California grid boundary (`df_ca`).

### Step 6B: Empirical Outlier Evaluation

The targeted diagnostic trace successfully isolated the explicit operational records responsible for the global distribution anomalies, revealing a sharp contrast between raw reporting corruptions and high-integrity regional telemetry:

* **Programmatic Corruption Vector (`WACM` / `WALC`):** The audit confirms that the multi-million megawatt spikes are localized entirely within out-of-state networks. Specifically, the `WACM` balancing authority experienced severe telemetry echoing on **March 10, 2026, at 09:00 PDT**, where corrupted placeholder metrics forced demand and generation channels above **2,576,000 MW**. Similarly, negative load bounds down to **-25,408 MW** are driven by midnight logging dropouts within the `WALC` network.
* **Contextual Scale Preservation (`PJM`):** In contrast to the corrupted spikes, high-end demand metrics trailing between **136,000 MW and 139,000 MW** for the `PJM` interconnection reflect true, high-integrity winter peak loads. This divergence confirms that applying an arbitrary, universal clipping threshold across the entire dataset would inadvertently compromise valid regional boundaries.
* **Validation of California's Import Identity (`CISO`):** Crucially for our project scope, the `total_interchange_mw` trace reveals that the absolute lowest values in the dataset belong to **CISO**, which consistently sustained massive negative flows ranging from **-10,930 MW to -11,236 MW** on January 15–16, 2026. Far from being data errors, these negative metrics represent real-world grid physics: capturing periods where California was aggressively importing over 11 GW of out-of-state power to keep its system balanced.

### ➔ Engineering Action Plan
These insights solidify our engineering approach: we will completely bypass universal row-clipping rules that risk removing valid out-of-state benchmarks. Instead, by moving straight to **Step 7** and executing our rigorous geographic filter, we will programmatically discard all multi-million MW `WACM` errors while preserving 100% of the true, high-integrity `CISO` transfer dynamics for our Tableau dashboards.

In [ ]:
# ============================================
# STEP 6B: INSPECT EXTREME VALUES (DIAGNOSTIC)
# ============================================

extreme_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]

review_fields = [
    "balancing_authority",
    "data_date",
    "hour_number",
    "local_time_pacific",
]

for col in extreme_cols:
    print("\n" + "=" * 75)
    print(f"🚨 EXTREME VALUE TRACE: {col.upper()}")
    print("=" * 75)

    # 1. Isolate and display the top 20 largest outliers
    print(f"📈 Top 20 Largest Values for {col}:")
    print("-" * 50)
    display(df[review_fields + [col]].sort_values(col, ascending=False).head(20))

    print("\n" + "." * 50)

    # 2. Isolate and display the top 20 smallest anomalies
    print(f"\n📉 Top 20 Smallest Values for {col}:")
    print("-" * 50)
    display(df[review_fields + [col]].sort_values(col, ascending=True).head(20))

## Step 6B Summary: Empirical Outlier Evaluation

The targeted diagnostic trace successfully isolated the explicit operational records responsible for the global distribution anomalies, revealing a sharp contrast between raw reporting corruptions and high-integrity regional telemetry:

* **Programmatic Corruption Vector (`WACM` / `WALC`):** The audit confirms that the multi-million megawatt spikes are localized entirely within out-of-state networks. Specifically, the `WACM` balancing authority experienced severe telemetry echoing on **March 10, 2026, at 09:00 PDT**, where corrupted placeholder metrics forced demand and generation channels above **2,576,000 MW**. Similarly, negative load bounds down to **-25,408 MW** are driven by midnight logging dropouts within the `WALC` network.
* **Contextual Scale Preservation (`PJM`):** In contrast to the corrupted spikes, high-end demand metrics trailing between **136,000 MW and 139,000 MW** for the `PJM` interconnection reflect true, high-integrity winter peak loads. This divergence confirms that applying an arbitrary, universal clipping threshold across the entire dataset would inadvertently compromise valid regional boundaries.
* **Validation of California's Import Identity (`CISO`):** Crucially for our project scope, the `total_interchange_mw` trace reveals that the absolute lowest values in the dataset belong to **CISO**, which consistently sustained massive negative flows ranging from **-10,930 MW to -11,236 MW** on January 15–16, 2026. Far from being data errors, these negative metrics represent real-world grid physics: capturing periods where California was aggressively importing over 11,000 MW of out-of-state power to keep its system balanced.



### Step 6C: California Isolation Control Check

A targeted validation gate was executed on a test subset containing **2,585 hourly records** exclusively for **CISO** (California Independent System Operator). This diagnostic control check explicitly verifies that the extreme multi-million megawatt corruptions and negative demand anomalies exposed in the global audit are entirely isolated out-of-state and do not compromise our primary target footprint.

Key analytical findings from this isolation trace include:
* **Empirical Range Normalization:** Within the CISO subset, actual demand behaves with strict physical consistency—ranging from a baseline night load of **18,748.00 MW** to an observed seasonal peak of **35,596.00 MW**, maintaining a tight statistical average of **24,332.09 MW**. This clean empirical footprint confirms that the raw federal dataset's multi-million MW tracking flaws are geographically confined.
* **Temporal Coherence of System Peaks:** Slicing the top 10 historical demand peaks reveals a precise structural clustering: peak system stress occurs uniformly between **17:00 and 20:00 Pacific Time** (5:00 PM to 8:00 PM) during mid-March 2026. This flawlessly captures the physical reality of the California "duck curve" peak, where residential consumer demand scales upward exactly as solar irradiance drop-offs force a massive ramp in local thermal generation and net imports.
* **Quantification of Structural Import Reliance:** The total interchange channel operates with a sustained negative posture, averaging **-5,634.73 MW** across the entire window. The maximum net import peaks stabilize between **-10,811.00 MW and -11,236.00 MW** during the pre-dawn hours (01:00 to 05:00 Pacific Time) of mid-January. Far from indicating a reporting flaw, these massive negative balances represent real-world grid orchestration: capturing periods where California aggressively imports over 11 GW of base power from neighboring regional networks to maintain localized frequency stability.

### ➔ Strategic Engineering Verdict
This control check provides absolute empirical validation for our final scope strategy. It proves that we do not need to implement complex or aggressive statistical clipping algorithms that risk masking valid regional dynamics. Instead, we can safely advance directly to **Section 6: Exploratory Data Analysis (EDA)**, where our localized geographical filters will naturally discard 100% of the out-of-state corruptions while preserving high-integrity California telemetry for our Tableau dashboards.

In [ ]:
# ============================================
# STEP 6C: CALIFORNIA BOUNDARY SCOPE VERIFICATION (CISO)
# ============================================

california_bas = ["CISO"]
df_ca_check = df[df["balancing_authority"].isin(california_bas)].copy()

print("🎯 TARGET FOOTPRINT ISOLATION")
print("=" * 65)
print(f"📋 CISO Test Subset Profile: {df_ca_check.shape[0]:,} hourly records")
print("=" * 65)

print("\n📊 CISO EMPIRICAL DISTRIBUTION CHECK")
print("-" * 65)
target_metrics = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]
display(df_ca_check[target_metrics].describe())
print("-" * 65)

print("\n📈 TRACE: TOP 10 HISTORICAL DEMAND PEAKS (CISO)")
print("-" * 65)
review_fields = [
    "balancing_authority",
    "data_date",
    "hour_number",
    "local_time_pacific",
]
display(
    df_ca_check[review_fields + target_metrics]
    .sort_values("demand_mw", ascending=False)
    .head(10)
)

print("\n📉 TRACE: MAXIMUM NET ENERGY IMPORT PEAKS (CISO)")
print("-" * 65)
display(
    df_ca_check[review_fields + target_metrics]
    .sort_values("total_interchange_mw", ascending=True)
    .head(10)
)

## Step 6C Summary: California Isolation Control Check

A targeted validation gate was executed on a test subset containing **2,585 hourly records** exclusively for **CISO** (California Independent System Operator). This diagnostic control check explicitly verifies that the extreme multi-million megawatt corruptions and negative demand anomalies exposed in the global audit are entirely isolated out-of-state and do not compromise our primary target footprint.

Key analytical findings from this isolation trace include:
* **Empirical Range Normalization:** Within the CISO subset, actual demand behaves with strict physical consistency—ranging from a baseline night load of **18,748.00 MW** to an observed seasonal peak of **35,596.00 MW**, maintaining a tight statistical average of **24,332.09 MW**. This clean empirical footprint confirms that the raw federal dataset's multi-million MW tracking flaws are geographically confined.
* **Temporal Coherence of System Peaks:** Slicing the top 10 historical demand peaks reveals a precise structural clustering: peak system stress occurs uniformly between **17:00 and 20:00 Pacific Time** (5:00 PM to 8:00 PM) during mid-March 2026. This flawlessly captures the physical reality of the California "duck curve" peak, where residential consumer demand scales upward exactly as solar irradiance drop-offs force a massive ramp in local thermal generation and net imports.
* **Quantification of Structural Import Reliance:** The total interchange channel operates with a sustained negative posture, averaging **-5,634.73 MW** across the entire window. The maximum net import peaks stabilize between **-10,811.00 MW and -11,236.00 MW** during the pre-dawn hours (01:00 to 05:00 Pacific Time) of mid-January. Far from indicating a reporting flaw, these massive negative balances represent real-world grid orchestration: capturing periods where California aggressively imports over 11 GW of base power from neighboring regional networks to maintain localized frequency stability.

### ➔ Strategic Engineering Verdict
This control check provides absolute empirical validation for our final scope strategy. It proves that we do not need to implement complex or aggressive statistical clipping algorithms that risk masking valid regional dynamics. Instead, by moving directly to **Step 7** and executing our formal multi-authority geographic filter, we will naturally dump 100% of the out-of-state corruptions while preserving the pure, high-integrity telemetry required for our Tableau Public data model.

### Step 7: Create the California Analysis Dataset

The raw federal EIA-930 file aggregates concurrent electricity data from multi-state networks across the entire nation. Because this project focuses exclusively on California's localized power grid infrastructure and operational constraints, the baseline dataset must be structurally isolated to our true regional footprint.

This step creates a dedicated regional analysis DataFrame named `df_ca` by applying a vectorized inclusion filter that captures all five major California balancing authorities: **BANC, CISO, IID, LDWP, and TIDC**. Partitioning this regional footprint away from the global matrix programmatically discards the multi-million megawatt out-of-state reporting errors and negative demand anomalies we exposed during our Step 6 audit, completely protecting our downstream data models and dashboard visualization axes.

While the full national dataset remains accessible within our repository for high-level macro comparisons, the core investigative analysis, localized feature engineering, and Tableau data models will derive exclusively from this high-integrity California subset.

In [ ]:
# ============================================
# STEP 7: CREATE THE CALIFORNIA ANALYSIS DATASET
# ============================================

# Define the complete geographic scope for California balancing authorities
california_scope = ["CISO", "BANC", "IID", "LDWP", "TIDC"]

# Apply a vectorized inclusion filter to build the unified California dataset
df_ca = df[df["balancing_authority"].isin(california_scope)].copy()

print("🎯 CALIFORNIA REGIONAL SCOPE ISOLATION")
print("=" * 65)
print(f"✅ California dataset created successfully.")
print(f"📋 Unified Footprint: {df_ca.shape[0]:,} rows × {df_ca.shape[1]} columns")
print("=" * 65)

print("\n🕒 Localized Temporal Boundaries (Pacific Time):")
print("-" * 65)
print(f"  ▪️ Timeline Activation: {df_ca['local_time_pacific'].min()}")
print(f"  ▪️ Timeline Termination: {df_ca['local_time_pacific'].max()}")
print("-" * 65)

print("\n📊 Multi-Authority Core Telemetry Distribution:")
print("-" * 65)
core_metrics = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]
display(df_ca[core_metrics].describe())
print("-" * 65)

## Step 7 Summary

The California regional analysis dataset was successfully constructed by filtering the global EIA-930 matrix down to our explicit geographic scope, capturing all five prominent California balancing authorities: **BANC, CISO, IID, LDWP, and TIDC**. This vectorized inclusion step created a unified, high-integrity sub-dataset consisting of **13,020 hourly records and 66 columns**, bridging a continuous operational timeline from **January 1, 2026, through April 20, 2026, in Pacific Time**.

The updated, multi-authority empirical distributions provide a clean regional baseline while highlighting the massive differences in utility operational scale now encapsulated in our data model:
* **Grid Scale & Demand Metrics:** Across the combined California footprint, system demand ranges from a baseline threshold of **0.00 MW** (capturing localized logging cutoffs or specific structural off-peaks in smaller districts) up to an absolute regional peak of **35,596.00 MW**. The pooled demand averages **5,839.07 MW** with a standard deviation of **9,334.15 MW**, dynamically capturing the vast difference between small municipal balancing pools like `TIDC` or `IID` and the massive statewide footprint managed by `CISO`.
* **Outlier Mitigation:** Crucially, the maximum values for actual demand (**35,596.00 MW**) and net generation (**31,201.00 MW**) remain firmly bounded within reasonable, real-world utility tolerances. This provides definitive proof that our geographical partitioning gate successfully purged 100% of the out-of-state, multi-million megawatt reporting corruptions exposed during our Step 6 global audit.
* **Structural Energy Interchange:** The pooled total interchange retains a strongly negative profile, operating at an aggregate statistical mean of **-1,321.01 MW** and reaching a maximum system import depth of **-11,236.00 MW**. This confirms that, as an integrated regional group, the California balancing authorities consistently rely on massive net power transfers from neighboring out-of-state interconnections to preserve localized frequency stability.

### ➔ Strategic Next Move
This high-integrity dataset is now officially certified clean, geographically isolated, and structurally stable. With all five California balancing authorities unified under a single, standardized framework, the data pipeline is fully prepared to advance to **Section 6: Exploratory Data Analysis (EDA)**. There, we will leverage interactive Plotly time-series visualizations to contrast daily load patterns and prepare our engineered metrics for ultimate dashboard ingestion.

## Section 6: Exploratory Data Analysis (EDA)

With a verified, high-integrity regional subset successfully isolated, the pipeline advances to Exploratory Data Analysis (EDA). This phase establishes our empirical baseline, validating visual distribution scaling before executing any downstream feature engineering.

Because California's electrical infrastructure spans municipal irrigation districts, metropolitan utilities, and a massive statewide transmission operator, a single collective visual will cause extreme scaling distortions. To prevent visual compression, our EDA pipeline sequentially executes a global scale comparison, an isolated multi-panel time-series matrix, and a localized temporal aggregation to track granular daily operational profiles across all five balancing authorities.

### Step 8: Recheck California-Only Demand Statistics

Before rendering visualizations, we run an isolated structural audit on our filtered `df_ca` dataset. This acts as an engineering confirmation gate—proving that our geographic filtering successfully purged the multi-million megawatt anomalies from out-of-state entities while cleanly mapping the relative operational boundaries of our five active target utilities.

In [ ]:
# ============================================
# STEP 8: RECHECK CALIFORNIA-ONLY DEMAND STATISTICS
# ============================================

core_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
]

print("🎯 REGIONAL SUBSET AUDIT GATE")
print("=" * 65)
print("📋 Unified California-Only Core Measure Summary:")
display(df_ca[core_cols].describe())
print("=" * 65)

print("\n⚙️ Core Measure Summary Breakdowns by Balancing Authority:")
display(
    df_ca.groupby("balancing_authority")[core_cols]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)
print("=" * 65)

### Step 8 Summary

The localized regional audit confirms a clean population of **13,020 hourly records** distributed across the five target balancing authorities. Total demand spans from **0.00 MW** up to an absolute regional peak of **35,596.00 MW**, verifying that all extreme out-of-state multi-million megawatt tracking anomalies have been successfully purged from our working dataframe.

The by-authority matrices reveal sharp scale disparities across the California grid:
* **Operational Scale Mapping:** `CISO` dominates the region with an intense average demand of **24,332.09 MW** and a maximum peak of **35,596.00 MW**. In contrast, municipal networks like `LDWP` and `BANC` sustain intermediate loads, while localized pools like `IID` and `TIDC` operate on much narrower margins.
* **Interchange Dynamics:** `CISO` exhibits a persistent negative posture, averaging **-5,634.73 MW**, indicating a heavy structural reliance on net out-of-state power imports to preserve frequency balance. Conversely, `IID` demonstrates a net-positive average interchange, acting as a regional generation exporter.

These scale differences confirm that executing a single, combined time-series chart will severely compress our smaller utilities. Consequently, these metrics will be evaluated using isolated panels in upcoming steps to preserve visualization fidelity.

### Step 9: Combined Scale Comparison by Balancing Authority

To visually demonstrate the massive scale differences across the California grid network, this step plots all five balancing authorities on a shared coordinate axis. This visualization highlights the true macro footprint of our data before we split the streams for granular review.

In [ ]:
# ============================================
# STEP 9: COMBINED SCALE COMPARISON BY BALANCING AUTHORITY
# ============================================

fig_compare = px.line(
    df_ca.sort_values(["balancing_authority", "local_time_pacific"]),
    x="local_time_pacific",
    y="demand_mw",
    color="balancing_authority",
    title="California Electricity Demand by Balancing Authority: Global Scale Comparison",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority",
    },
)

fig_compare.update_layout(template="plotly_white")
fig_compare.show()

### Step 9 Summary

The combined time-series plot visually confirms our empirical scaling hypothesis. Because `CISO` operates on a scale an order of magnitude larger than its neighboring networks, it completely monopolizes the upper bounds of the shared y-axis. 

While this visualization perfectly illustrates the relative dominance of `CISO` across the statewide energy footprint, it introduces severe visual compression along the baseline. Smaller balancing pools like `IID` and `TIDC` are flattened into unreadable ribbons near the bottom of the canvas, rendering their localized daily demand loops completely invisible. This chart serves as an excellent structural justification for our next step: transitioning to an independent panel framework where each network can scale dynamically on its own axis.

### Step 10: Panel View with Clean Average Demand Labels

To untangle the compressed baselines, this step builds an isolated, stacked multi-panel grid visualization. By allowing each balancing authority to utilize its own independent y-axis bounds while referencing a shared chronological x-axis, we can trace discrete, network-specific behavior over time.

In [ ]:
# ============================================
# STEP 10: PANEL VIEW WITH AUTOMATED REFERENCE METRICS
# ============================================

# Establish top-to-bottom layout visualization order
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Sort the dataset to guarantee sequential time-series rendering lines
df_ca_sorted = df_ca.sort_values(["balancing_authority", "local_time_pacific"])

# Compute baseline mean metrics for horizontal target overlays
avg_demand_by_ba = df_ca_sorted.groupby(
    "balancing_authority", as_index=False
).agg(avg_demand_mw=("demand_mw", "mean"))

# Instantiate the stacked facet row panel structure
fig_panel = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Hourly Demand Profiles by California Balancing Authority with Localized Baseline Mean Overlays",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority",
    },
    height=1050,
)

# Strip out structural pandas labeling string prefixes ("balancing_authority=")
fig_panel.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Break the coordinate axis lock to enable independent vertical scaling per panel
fig_panel.update_yaxes(matches=None)

n_panels = len(ba_order)

# Dynamically overlay local baseline mean indicators and text callouts
for ba in ba_order:
    avg_value = avg_demand_by_ba.loc[
        avg_demand_by_ba["balancing_authority"] == ba, "avg_demand_mw"
    ].iloc[0]

    # Convert native top-down indexing into Plotly's bottom-up row grid numbering
    row_number = n_panels - ba_order.index(ba)

    # Inject horizontal baseline reference indicators
    fig_panel.add_hline(
        y=avg_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1,
    )

    # Superimpose high-visibility metric annotation callout cards
    fig_panel.add_annotation(
        text=f"{ba} Baseline Mean: {avg_value:,.0f} MW",
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.90,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="red",
        borderwidth=1,
        font=dict(size=11),
        row=row_number,
        col=1,
    )

fig_panel.update_layout(template="plotly_white")
fig_panel.show()

### Step 10 Summary

The multi-panel grid view untangles our visualization canvas, bringing hidden utility behaviors to light. By decoupling the vertical scales, we preserve the distinct morphological identity of each utility's demand curve while maintaining perfect historical synchronization across the shared horizontal time-axis.

The explicit red baseline mean indicators immediately expose highly distinct operational signatures:
* **Diurnal Cyclic Synchronization:** `IID` and `TIDC` present exceptionally clean, smooth sinusoidal loops—capturing classic daily customer behavioral shifts at a localized municipal scale.
* **Data Logging Discontinuities:** `LDWP` displays extreme, flat-lined drops near zero at both the absolute boundaries of the temporal spectrum, capturing an structural logging intermission rather than a physical black sky event.
* **High-Frequency Deviations:** `BANC` exhibits a collection of sharp, vertical telemetry spikes reaching up toward **9,770 MW** that radically overshoot its standard baseline mean, identifying localized system stress hours that warrant deeper investigative auditing.

This visual confirmation proves that evaluating raw, unscaled megawatt loads is an unequal mechanism for identifying statewide grid stress. To make fair comparisons later, our metrics must be normalized against each network's unique operational capacity.

### Step 11: Create Daily Demand Trend Views

While our hourly panel matrix tracks real-time grid dynamics, the high density of raw hourly parameters across a multi-month footprint can introduce visual noise. To uncover long-term trends and cyclical patterns, this step executes an interim temporal aggregation—converting our granular data into a streamlined daily profile tracking both daily average and absolute daily peak demand.

In [ ]:
# ============================================
# STEP 11: CREATE DAILY DEMAND TREND VIEWS
# ============================================

# Compress hourly time-series into daily aggregates using localized time anchors
daily_demand = (
    df_ca.assign(local_date=df_ca["local_time_pacific"].dt.date)
    .groupby(["local_date", "balancing_authority"], as_index=False)
    .agg(
        daily_avg_demand_mw=("demand_mw", "mean"),
        daily_peak_demand_mw=("demand_mw", "max"),
    )
)

# Cast the aggregated objects back into datetime parameters for seamless plotting
daily_demand["local_date"] = pd.to_datetime(daily_demand["local_date"])

print("✅ Daily demand temporal aggregation complete.")
print(f"📋 Compressed Footprint: {daily_demand.shape[0]} daily summary records")
print("-" * 65)

print("\n⭐ Leading Aggregated Data Points Preview:")
display(daily_demand.head())
print("-" * 65)

print("\n📊 Daily Aggregation Distribution Breakdown by Network:")
display(
    daily_demand.groupby("balancing_authority")[
        ["daily_avg_demand_mw", "daily_peak_demand_mw"]
    ]
    .agg(["mean", "min", "max"])
    .round(2)
)
print("-" * 65)

### Step 11A: Daily Peak Demand Chart Visualization

While daily averages are excellent for tracking baseline load shifts, grid operators evaluate physical reliability based on maximum system stress. This step generates an interactive time-series visualization using Plotly Express to trace the absolute peak electricity demand reached within each balancing authority for every calendar day.

In [ ]:
# ============================================
# STEP 11A: DAILY PEAK DEMAND CHART
# ============================================

fig_daily_peak = px.line(
    daily_demand,
    x="local_date",
    y="daily_peak_demand_mw",
    color="balancing_authority",
    title="Daily Peak Electricity Demand Profiles by California Balancing Authority",
    labels={
        "local_date": "Date",
        "daily_peak_demand_mw": "Daily Peak Demand (MW)",
        "balancing_authority": "Balancing Authority",
    },
)

# Apply standard white canvas layout for high-end presentation matching
fig_daily_peak.update_layout(
    template="plotly_white",
    xaxis_title="Timeline (Date)",
    yaxis_title="Absolute Peak Demand (MW)",
    hovermode="x unified",
)

fig_daily_peak.show()

### Step 11A Summary

The daily peak demand chart provides a clean, macro-level perspective on system stress across the continuous multi-month timeline, successfully removing the visual density of our 13,020 raw hourly parameters:

* **Isolating Top-End Volatility:** Unlike daily averages which smooth out consumption spikes, this chart successfully highlights the single hour of maximum stress achieved each day. For instance, the dramatic mid-March peak within the `CISO` network stretching to **35,596.00 MW** stands out instantly, allowing reviewers to pinpoint periods of extreme operational pressure at a glance.
* **Persistent Scale Compression Rationale:** Even at a daily resolution, the global scale differences across the California balancing authorities remain distinct. The massive statewide envelope of `CISO` still forces municipal networks like `IID` and `TIDC` down into tight baseline bands near the bottom of the canvas.

This chart confirms that while daily aggregates are superior for dashboard executive views, raw peak load figures remain an unfair mechanism for cross-network reliability assessments. This delivers the definitive empirical foundation for our next phase: mapping these raw daily parameters into a standardized **Grid Stress Index** where each independent network is normalized directly against its own maximum operating limits.

### Step 11B: Daily Peak Demand Panel View with Mean and Median Overlays

To resolve the scale compression present in the global peak chart, this step constructs an independent multi-panel grid visualization. Each authority scales dynamically on its own vertical axis, synchronized across a shared chronological x-axis. 

To introduce advanced diagnostic value, each panel programmatically calculates and overlays two distinct statistical markers: a **red dashed line for the daily peak mean** and a **green dotted line for the daily peak median**. Tracing the divergence between these two central tendency metrics allows us to immediately distinguish between structured, repetitive load cycles and volatile, anomaly-driven peak events.

In [ ]:
# ============================================
# STEP 11B: DAILY PEAK DEMAND PANEL DIAGNOSTIC
# ============================================

# Define visual order from top to bottom matching our standard structural layout
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate mean and median daily peak metrics to establish central tendency divergence
daily_peak_stats = daily_demand.groupby(
    "balancing_authority", as_index=False
).agg(
    mean_daily_peak_mw=("daily_peak_demand_mw", "mean"),
    median_daily_peak_mw=("daily_peak_demand_mw", "median"),
)

# Instantiate the stacked facet row panel structure
fig_daily_peak_panel = px.line(
    daily_demand.sort_values(["balancing_authority", "local_date"]),
    x="local_date",
    y="daily_peak_demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Daily Peak Demand Profiles by California Balancing Authority: Multi-Panel Statistical Audit",
    labels={
        "local_date": "Date",
        "daily_peak_demand_mw": "Daily Peak Demand (MW)",
        "balancing_authority": "Balancing Authority",
    },
    height=1050,
)

# Strip out structural pandas labeling string prefixes ("balancing_authority=")
fig_daily_peak_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Break coordinate axis lock to enable independent vertical scaling per utility panel
fig_daily_peak_panel.update_yaxes(matches=None)

n_panels = len(ba_order)

for ba in ba_order:
    # Convert native top-down indexing into Plotly's bottom-up row grid numbering
    row_number = n_panels - ba_order.index(ba)

    mean_value = daily_peak_stats.loc[
        daily_peak_stats["balancing_authority"] == ba, "mean_daily_peak_mw"
    ].iloc[0]

    median_value = daily_peak_stats.loc[
        daily_peak_stats["balancing_authority"] == ba, "median_daily_peak_mw"
    ].iloc[0]

    # Overlay horizontal Mean reference boundary line
    fig_daily_peak_panel.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1,
    )

    # Overlay horizontal Median reference boundary line
    fig_daily_peak_panel.add_hline(
        y=median_value,
        line_dash="dot",
        line_color="green",
        line_width=1.5,
        row=row_number,
        col=1,
    )

    # Superimpose high-visibility dual-metric summary cards
    fig_daily_peak_panel.add_annotation(
        text=(
            f"<b>{ba} Peak Audit</b><br>"
            f"Mean: {mean_value:,.0f} MW<br>"
            f"Median: {median_value:,.0f} MW"
        ),
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.92,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=10),
        row=row_number,
        col=1,
    )

fig_daily_peak_panel.update_layout(template="plotly_white", hovermode="x unified")
fig_daily_peak_panel.show()

### Step 11B Summary

The decoupled multi-panel peak demand chart delivers a powerful tool for discovering data anomalies, revealing critical skewness and operational signatures across each balancing authority's top-end boundary metrics:

* **Statistical Skewness and Shock Identification:** Evaluating the distance between the red (mean) and green (median) horizontal markers serves as a proxy for tracking volatility. For example, within the `BANC` network, the daily peak mean pulls noticeably upward from the median line. This positive statistical skew confirms that BANC's distribution profile is driven by occasional, high-impact demand shocks that overshoot normal daily cycles.
* **Sinusoidal Distribution Integrity:** Conversely, networks like `IID` and `TIDC` show an almost perfect convergence of their mean and median metrics. This alignment mathematically confirms a balanced, highly symmetrical distribution—proving that their peak system stress behaves as a steady, predictable function of daily weather and behavioral patterns.
* **Lineage Tracking of Missing Logs:** The panel framework preserves a clear record of data gaps. The prolonged drops near zero within `LDWP` remain perfectly visible on its independent timeline, indicating a structural logging lapse rather than an actual grid emergency.

### ➔ Strategic Rationale for Feature Engineering
This diagnostic chart provides the final statistical justification for our next major notebook phase: **Section 7: Stress Index Feature Engineering**. Because a $500\text{ MW}$ spike represents a minor variation for `CISO` but a major capacity shock for `TIDC`, raw peak values cannot serve as a reliable baseline for statewide comparison. We must transition immediately to normalizations—translating these raw parameters into a uniform, localized percentage framework to identify true system stress across all five networks.

### Step 14: Additional EDA for Grid Behavior

The previous demand visualizations showed that California balancing authorities differ strongly in scale, peak behavior, and daily demand patterns. Before creating the stress index, the analysis now examines additional operational signals that help explain grid behavior beyond demand alone.

This section reviews forecast accuracy, the relationship between demand and net generation, interchange patterns, hourly demand profiles, heatmap-based time patterns, and day-of-week demand variation. These views provide the practical foundation for designing a stress index that reflects multiple grid conditions rather than relying on raw demand alone.

In [ ]:
# ============================================
# STEP 14A: COMPARE ACTUAL DEMAND AND DEMAND FORECAST
# ============================================

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create panel chart comparing actual demand and forecasted demand
fig_demand_vs_forecast = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y=["demand_mw", "demand_forecast_mw"],
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Actual Demand vs Forecasted Demand by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "value": "Megawatts (MW)",
        "variable": "Series"
    },
    height=1050
)

# Clean facet labels
fig_demand_vs_forecast.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Allow each balancing authority to use its own y-axis scale
fig_demand_vs_forecast.update_yaxes(matches=None)

fig_demand_vs_forecast.show()

In [ ]:
# ============================================
# STEP 14A CHECK: FORECAST ERROR SUMMARY
# ============================================

# Create forecast error fields
df_ca["forecast_error_mw"] = df_ca["demand_mw"] - df_ca["demand_forecast_mw"]
df_ca["abs_forecast_error_mw"] = df_ca["forecast_error_mw"].abs()

# Summarize forecast error by balancing authority
forecast_error_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("forecast_error_mw", "count"),
        mean_error_mw=("forecast_error_mw", "mean"),
        median_error_mw=("forecast_error_mw", "median"),
        mean_abs_error_mw=("abs_forecast_error_mw", "mean"),
        max_abs_error_mw=("abs_forecast_error_mw", "max")
    )
    .round(2)
)

print("Forecast error summary by balancing authority:")
display(forecast_error_summary)

#### Step 14A Summary

The actual demand and forecast comparison shows that forecast quality differs across California balancing authorities. IID and TIDC have very small forecast errors, with mean absolute errors of about **10.87 MW** and **8.83 MW**, respectively. Their actual and forecasted demand lines closely overlap for most of the observed period.

CISO shows the largest forecast gap. Its mean error is about **2,234 MW**, and its mean absolute error is about **2,351 MW**. This means actual CISO demand was generally higher than forecasted demand during the study period. CISO also has the largest maximum absolute forecast error at **10,573 MW**, which suggests that some periods had substantial planning deviation.

BANC has a small average forecast error, but its maximum absolute error reaches **7,616 MW**, likely due to isolated spike behavior rather than consistent forecast mismatch. LDWP shows moderate forecast error, with a mean absolute error of about **462 MW**, and its forecast line does not fully capture some demand shifts, especially near the beginning and end of the observed period.

This step shows that forecast error is an important operational signal. For the later stress index, forecast deviation should be considered alongside demand level, because high demand is more concerning when it is also under-forecast or poorly anticipated.

#### Step 14B: Compare Demand and Net Generation

Demand is compared with net generation to examine whether local generation is moving with electricity use. When demand is higher than net generation, the difference is often supported by interchange with neighboring systems.

This comparison helps explain grid balance. It also supports the stress index design by showing that high demand alone is not the full story. A high-demand period may be more operationally important when local generation is low or when the authority depends more heavily on interchange.

In [ ]:
# ============================================
# STEP 14B: COMPARE DEMAND AND NET GENERATION
# ============================================

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create panel chart comparing demand and net generation
fig_generation_vs_demand = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y=["demand_mw", "net_generation_mw"],
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Demand vs Net Generation by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "value": "Megawatts (MW)",
        "variable": "Series"
    },
    height=1050
)

# Clean facet labels
fig_generation_vs_demand.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Allow each authority to use its own y-axis scale
fig_generation_vs_demand.update_yaxes(matches=None)

fig_generation_vs_demand.show()

In [ ]:
# ============================================
# STEP 14B CHECK: GENERATION-DEMAND GAP SUMMARY
# ============================================

# Positive gap means demand is greater than net generation.
# This can indicate reliance on imports or other balancing mechanisms.
df_ca["generation_gap_mw"] = df_ca["demand_mw"] - df_ca["net_generation_mw"]

generation_gap_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("generation_gap_mw", "count"),
        mean_generation_gap_mw=("generation_gap_mw", "mean"),
        median_generation_gap_mw=("generation_gap_mw", "median"),
        max_generation_gap_mw=("generation_gap_mw", "max"),
        min_generation_gap_mw=("generation_gap_mw", "min")
    )
    .round(2)
)

print("Generation-demand gap summary by balancing authority:")
display(generation_gap_summary)

#### Step 14B Summary

The demand and net generation comparison shows that California balancing authorities differ not only in demand scale, but also in how closely local generation tracks demand.

CISO has the largest generation-demand gap. Its average gap is about **8,816 MW**, meaning demand is consistently higher than reported net generation during the observed period. This supports the earlier interchange finding that CISO often depends on power flows from outside its balancing authority to meet demand.

LDWP and BANC also show positive average generation gaps, about **780 MW** and **447 MW**, respectively. This indicates that demand is usually above net generation for these authorities as well, although at much smaller scales than CISO.

IID is different. Its average generation-demand gap is about **-290 MW**, meaning reported net generation is often higher than demand. This suggests that IID may more often operate as a net exporter or have generation patterns that exceed local demand during many hours. TIDC shows a smaller positive gap of about **92 MW**, indicating a more modest difference between demand and net generation.

This step confirms that supply balance is an important part of California grid behavior. A later stress index should not rely only on demand level. It should also account for generation-demand gap and interchange conditions, because high demand becomes more operationally important when local generation is not keeping pace.

#### Step 14C: Review Total Interchange

Total interchange shows whether a balancing authority is exchanging power with neighboring systems. Negative values generally indicate net imports, while positive values indicate net exports.

Reviewing interchange helps explain whether demand is being supported by local generation or by power flows from outside the balancing authority. This is especially important after the demand and net generation comparison, because generation gaps are often balanced through interchange.

In [ ]:
# ============================================
# STEP 14C: REVIEW TOTAL INTERCHANGE
# ============================================

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create interchange chart by balancing authority
fig_interchange = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y="total_interchange_mw",
    color="balancing_authority",
    title="Total Interchange Over Time by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "total_interchange_mw": "Total Interchange (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

# Add zero reference line to separate net import and net export periods
fig_interchange.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig_interchange.show()

In [ ]:
# ============================================
# STEP 14C CHECK: INTERCHANGE SUMMARY
# ============================================

interchange_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("total_interchange_mw", "count"),
        mean_interchange_mw=("total_interchange_mw", "mean"),
        median_interchange_mw=("total_interchange_mw", "median"),
        min_interchange_mw=("total_interchange_mw", "min"),
        max_interchange_mw=("total_interchange_mw", "max")
    )
    .round(2)
)

print("Interchange summary by balancing authority:")
display(interchange_summary)

#### Step 14C Summary

The total interchange chart shows clear differences in how California balancing authorities exchange power with neighboring systems. The zero reference line separates net import periods from net export periods.

CISO has the largest negative interchange pattern, with an average total interchange of about **-5,635 MW** and a minimum value of **-11,236 MW**. This means CISO is frequently a net importer during the observed period. This finding is consistent with the earlier generation-demand gap analysis, where CISO demand was consistently higher than reported net generation.

BANC, LDWP, and TIDC also show negative average interchange values, suggesting net import behavior on average. LDWP’s average interchange is about **-780 MW**, BANC’s is about **-447 MW**, and TIDC’s is about **-90 MW**. IID is different, with a positive average interchange of about **301 MW**, suggesting that it often exports more power than it imports during the study period.

This step confirms that interchange is a useful operational signal for the stress index. High demand may be more important when it occurs alongside large negative interchange, because that combination can indicate heavier reliance on external power flows.

#### Step 14D: Review Average Demand by Hour of Day

Average demand by hour of day shows the typical daily load shape for each California balancing authority. This view helps identify the hours when electricity demand is usually highest and provides useful context for stress index design.

Because balancing authorities operate at different scales, this step uses both a combined chart and a panel chart. The combined chart helps compare statewide scale, while the panel chart makes each authority’s daily shape easier to read.

In [ ]:
# ============================================
# STEP 14D: REVIEW AVERAGE DEMAND BY HOUR OF DAY
# ============================================

# Add hour-of-day feature for time-based demand profiling
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Calculate average demand by authority and hour of day
hourly_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(avg_demand_mw=("demand_mw", "mean"))
)

print("Hourly demand profile created.")
print("Shape:", hourly_profile.shape)

display(hourly_profile.head())

# Combined hourly profile chart
fig_hourly_profile = px.line(
    hourly_profile,
    x="hour_of_day",
    y="avg_demand_mw",
    color="balancing_authority",
    title="Average Demand by Hour of Day by California Balancing Authority",
    labels={
        "hour_of_day": "Hour of Day",
        "avg_demand_mw": "Average Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_hourly_profile.show()

In [ ]:
# ============================================
# STEP 14D PANEL VIEW: AVERAGE DEMAND BY HOUR OF DAY
# WITH MEAN AND MEDIAN REFERENCE LINES
# ============================================

# Create hourly profile
hourly_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(avg_demand_mw=("demand_mw", "mean"))
)

# Compute mean and median of the hourly profile for each authority
hourly_profile_stats = (
    hourly_profile
    .groupby("balancing_authority", as_index=False)
    .agg(
        mean_hourly_profile_mw=("avg_demand_mw", "mean"),
        median_hourly_profile_mw=("avg_demand_mw", "median")
    )
)

ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

fig_hourly_profile_panel = px.line(
    hourly_profile,
    x="hour_of_day",
    y="avg_demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Average Demand by Hour of Day: Panel View with Mean and Median Reference Lines",
    labels={
        "hour_of_day": "Hour of Day",
        "avg_demand_mw": "Average Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_hourly_profile_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_hourly_profile_panel.update_yaxes(matches=None)

n_panels = len(ba_order)

for ba in ba_order:
    row_number = n_panels - ba_order.index(ba)

    mean_value = hourly_profile_stats.loc[
        hourly_profile_stats["balancing_authority"] == ba,
        "mean_hourly_profile_mw"
    ].iloc[0]

    median_value = hourly_profile_stats.loc[
        hourly_profile_stats["balancing_authority"] == ba,
        "median_hourly_profile_mw"
    ].iloc[0]

    fig_hourly_profile_panel.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1
    )

    fig_hourly_profile_panel.add_hline(
        y=median_value,
        line_dash="dot",
        line_color="green",
        line_width=1.5,
        row=row_number,
        col=1
    )

    fig_hourly_profile_panel.add_annotation(
        text=(
            f"{ba}<br>"
            f"Mean: {mean_value:,.0f} MW<br>"
            f"Median: {median_value:,.0f} MW"
        ),
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.92,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=10),
        row=row_number,
        col=1
    )

fig_hourly_profile_panel.show()

#### Step 14D2: Median Hourly Demand Profile with IQR Band

The previous hourly profile chart showed average demand by hour of day. This enhanced view uses the median and interquartile range, which gives a more robust picture of typical daily demand behavior.

The median line shows the typical demand level for each hour. The shaded band shows the interquartile range, from the 25th percentile to the 75th percentile. This band captures the middle 50% of observed demand values for each hour and helps show how much demand varies during different parts of the day.

This approach is useful because it is less sensitive to unusual spikes than a simple average line. It provides a stronger foundation for stress-index design by showing not only when demand is typically high, but also when hourly demand is more variable.

In [ ]:
# ============================================
# STEP 14D2: MEDIAN HOURLY DEMAND PROFILE WITH IQR BAND
# ============================================

# Create hour-of-day field if it does not already exist
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate median and interquartile range by authority and hour
hourly_iqr_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        median_demand_mw=("demand_mw", "median"),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75))
    )
)

print("Hourly median and IQR profile created.")
print("Shape:", hourly_iqr_profile.shape)
display(hourly_iqr_profile.head())

In [ ]:
# ============================================
# STEP 14D2: MEDIAN HOURLY DEMAND PROFILE WITH IQR BAND
# AND MEAN REFERENCE LINE
# ============================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create hour-of-day field if it does not already exist
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate hourly summary statistics by authority and hour
hourly_iqr_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        median_demand_mw=("demand_mw", "median"),
        mean_demand_mw=("demand_mw", "mean"),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75))
    )
)

print("Hourly median, mean, and IQR profile created.")
print("Shape:", hourly_iqr_profile.shape)
display(hourly_iqr_profile.head())


# ============================================
# STEP 14D2 VISUAL: MEDIAN HOURLY PROFILE WITH IQR BAND
# AND MEAN REFERENCE LINE
# ============================================

# Create one row per balancing authority
fig_iqr = make_subplots(
    rows=len(ba_order),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=ba_order
)

for row_num, ba in enumerate(ba_order, start=1):
    ba_data = (
        hourly_iqr_profile[
            hourly_iqr_profile["balancing_authority"] == ba
        ]
        .sort_values("hour_of_day")
    )

    # Upper IQR boundary
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["q75_demand_mw"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
            name=f"{ba} 75th percentile"
        ),
        row=row_num,
        col=1
    )

    # Lower IQR boundary with fill to previous trace
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["q25_demand_mw"],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(255, 165, 0, 0.25)",
            showlegend=(row_num == 1),
            name="IQR band: 25th-75th percentile",
            hoverinfo="skip"
        ),
        row=row_num,
        col=1
    )

    # Mean demand line
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["mean_demand_mw"],
            mode="lines",
            line=dict(width=2, color="red", dash="dash"),
            showlegend=(row_num == 1),
            name="Mean demand",
            hovertemplate=(
                "Hour: %{x}<br>"
                "Mean Demand: %{y:,.0f} MW"
                "<extra></extra>"
            )
        ),
        row=row_num,
        col=1
    )

    # Median demand line
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["median_demand_mw"],
            mode="lines+markers",
            line=dict(width=2.5, color="rgb(31, 78, 121)"),
            marker=dict(size=4, color="rgb(31, 78, 121)"),
            showlegend=(row_num == 1),
            name="Median demand",
            hovertemplate=(
                "Hour: %{x}<br>"
                "Median Demand: %{y:,.0f} MW"
                "<extra></extra>"
            )
        ),
        row=row_num,
        col=1
    )

    # Add panel-specific y-axis title
    fig_iqr.update_yaxes(
        title_text=f"{ba}<br>MW",
        row=row_num,
        col=1
    )

# Layout formatting
fig_iqr.update_layout(
    title="Median Hourly Demand Profile with IQR Band and Mean Line by California Balancing Authority",
    height=1100,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Apply x-axis settings to every subplot
for row_num in range(1, len(ba_order) + 1):
    fig_iqr.update_xaxes(
        range=[-0.2, 23.2],
        tickmode="linear",
        tick0=0,
        dtick=1,
        row=row_num,
        col=1
    )

# Add x-axis title to the bottom panel only
fig_iqr.update_xaxes(
    title_text="Hour of Day",
    row=len(ba_order),
    col=1
)

fig_iqr.show()

#### Step 14D2 Summary

The median hourly demand profile with an IQR band and mean line provides a strong view of daily demand behavior across California balancing authorities. The dark blue line shows the median hourly demand, the amber band shows the interquartile range, and the red dashed line shows the mean hourly demand.

This combination helps distinguish three things at once: the typical demand level, the middle range of variation, and the effect of higher-demand values on the average. Where the mean line sits above the median line, the hourly demand pattern may be slightly right-skewed, meaning higher values are pulling the average upward.

The chart confirms that demand is strongly time-dependent across authorities. Most authorities show lower demand in early morning hours and higher demand later in the day. The IQR band also shows that some hours are more variable than others, which is useful for identifying when demand behavior is less stable.

This view is especially useful for stress-index design because it combines typical behavior, variation, and average level in one chart. It helps separate ordinary hourly demand structure from unusually high or more variable demand periods.

#### Step 14E: Demand Heatmap by Date and Hour

The demand heatmap shows how electricity demand varies by date and hour of day. This view is useful because line charts can hide repeated daily patterns. A heatmap makes it easier to identify recurring high-demand hours, unusual days, and periods when demand pressure is concentrated.

Because the project covers statewide California balancing authorities with different scales, this step begins with a focused CISO heatmap. CISO is the largest California balancing authority, so it provides the clearest view of the main statewide demand pattern. The broader California analysis still includes BANC, IID, LDWP, and TIDC in the comparison charts and summary tables.

In [ ]:
# ============================================
# STEP 14E: DEMAND HEATMAP BY DATE AND HOUR
# ============================================

# Create a CISO-only subset for a focused heatmap
df_ciso = df_ca[df_ca["balancing_authority"] == "CISO"].copy()

# Create date and hour fields for heatmap structure
df_ciso["local_date"] = df_ciso["local_time_pacific"].dt.date
df_ciso["hour_of_day"] = df_ciso["local_time_pacific"].dt.hour

# Pivot data so dates are rows and hours are columns
heatmap_data = (
    df_ciso.pivot_table(
        index="local_date",
        columns="hour_of_day",
        values="demand_mw",
        aggfunc="mean"
    )
)

print("CISO heatmap data created.")
print("Shape:", heatmap_data.shape)
display(heatmap_data.head())

fig_heatmap = px.imshow(
    heatmap_data,
    aspect="auto",
    title="CISO Demand Heatmap by Date and Hour",
    labels={
        "x": "Hour of Day",
        "y": "Date",
        "color": "Demand (MW)"
    },
    color_continuous_scale="YlOrRd"
)

fig_heatmap.update_xaxes(
    tickmode="linear",
    tick0=0,
    dtick=1
)

fig_heatmap.show()

#### Step 14E2: Demand Heatmaps for All California Balancing Authorities

The CISO heatmap shows the largest California authority, but the statewide project also includes BANC, IID, LDWP, and TIDC. Because these authorities operate at very different demand scales, separate heatmaps are created for each balancing authority.

Separate heatmaps allow each authority’s date-and-hour demand pattern to be reviewed within its own range. This prevents CISO’s larger scale from hiding meaningful patterns in smaller authorities.

In [ ]:
# ============================================
# STEP 14E2: DEMAND HEATMAPS FOR ALL CALIFORNIA BALANCING AUTHORITIES
# ============================================

# Create helper date and hour fields
df_ca["local_date"] = df_ca["local_time_pacific"].dt.date
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

for ba in ba_order:
    ba_heatmap_data = (
        df_ca[df_ca["balancing_authority"] == ba]
        .pivot_table(
            index="local_date",
            columns="hour_of_day",
            values="demand_mw",
            aggfunc="mean"
        )
    )

    print(f"{ba} heatmap data shape:", ba_heatmap_data.shape)

    fig_ba_heatmap = px.imshow(
        ba_heatmap_data,
        aspect="auto",
        title=f"{ba} Demand Heatmap by Date and Hour",
        labels={
            "x": "Hour of Day",
            "y": "Date",
            "color": "Demand (MW)"
        },
        color_continuous_scale="YlOrRd"
    )

    fig_ba_heatmap.update_xaxes(
        tickmode="linear",
        tick0=0,
        dtick=1
    )

    fig_ba_heatmap.show()

#### Step 14E3: Normalized Demand Heatmaps by Balancing Authority

Raw demand heatmaps are useful for understanding each authority’s absolute demand pattern. However, raw megawatt values are not directly comparable across authorities because CISO is much larger than BANC, IID, LDWP, and TIDC.

To make the heatmaps comparable, demand is normalized within each balancing authority using percentile rank. A value near 1.00 means demand is high relative to that authority’s own observed range, while a value near 0.00 means demand is low relative to that authority’s own history.

This view is especially useful before building the stress index because the stress index should compare abnormal pressure within each authority, not simply reward the largest authority for having the largest raw demand.

In [ ]:
# ============================================
# STEP 14E3: NORMALIZED DEMAND HEATMAPS BY BALANCING AUTHORITY
# ============================================

# Create a normalized copy for heatmap comparison
df_ca_heat = df_ca.copy()

df_ca_heat["local_date"] = df_ca_heat["local_time_pacific"].dt.date
df_ca_heat["hour_of_day"] = df_ca_heat["local_time_pacific"].dt.hour

# Percentile-rank demand within each balancing authority
df_ca_heat["demand_percentile_within_ba"] = (
    df_ca_heat
    .groupby("balancing_authority")["demand_mw"]
    .rank(pct=True)
)

for ba in ba_order:
    ba_norm_heatmap_data = (
        df_ca_heat[df_ca_heat["balancing_authority"] == ba]
        .pivot_table(
            index="local_date",
            columns="hour_of_day",
            values="demand_percentile_within_ba",
            aggfunc="mean"
        )
    )

    print(f"{ba} normalized heatmap data shape:", ba_norm_heatmap_data.shape)

    fig_norm_heatmap = px.imshow(
        ba_norm_heatmap_data,
        aspect="auto",
        title=f"{ba} Normalized Demand Heatmap by Date and Hour",
        labels={
            "x": "Hour of Day",
            "y": "Date",
            "color": "Demand Percentile"
        },
        color_continuous_scale="YlOrRd",
        zmin=0,
        zmax=1
    )

    fig_norm_heatmap.update_xaxes(
        tickmode="linear",
        tick0=0,
        dtick=1
    )

    fig_norm_heatmap.show()

#### Step 14E Summary

The normalized demand heatmaps show when each California balancing authority experienced high demand relative to its own observed range. This is more useful for cross-authority comparison than raw megawatt values because CISO operates at a much larger scale than the other authorities.

Across the normalized heatmaps, higher relative demand is generally concentrated later in the day, especially from the afternoon into the evening. This confirms that demand pressure is strongly tied to hour-of-day patterns. The heatmaps also show that several authorities experienced stronger relative demand during mid-to-late March, which matches the peak behavior seen in earlier line charts.

The normalized view is especially important for smaller authorities such as IID and TIDC. In raw megawatt terms, these authorities appear small compared with CISO. But in normalized terms, their own high-demand periods become visible. This supports a fairer statewide analysis because grid stress should be measured relative to each authority’s normal operating range, not only by absolute demand.

These heatmaps provide direct support for the stress index design. A useful stress index should include within-authority normalization so that high-demand conditions in smaller authorities are not hidden by CISO’s larger scale.

#### Step 14F: Demand Distribution by Day of Week

Demand can vary by day of week because business activity, industrial activity, household routines, and weekend behavior differ across the week. This step examines whether demand levels are systematically different across weekdays and weekends.

A boxplot is used because it shows the median, spread, and high-demand outliers for each day. This provides another view of demand variability before the stress index is created.

In [ ]:
# ============================================
# STEP 14F: DEMAND DISTRIBUTION BY DAY OF WEEK
# ============================================

# Add day-of-week name for weekly demand pattern analysis
df_ca["day_name"] = df_ca["local_time_pacific"].dt.day_name()

# Set calendar order for readability
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

fig_dow = px.box(
    df_ca,
    x="day_name",
    y="demand_mw",
    color="balancing_authority",
    category_orders={"day_name": day_order},
    title="Demand Distribution by Day of Week and Balancing Authority",
    labels={
        "day_name": "Day of Week",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_dow.show()

In [ ]:
# ============================================
# STEP 14F PANEL VIEW: DEMAND DISTRIBUTION BY DAY OF WEEK
# ============================================

fig_dow_panel = px.box(
    df_ca,
    x="day_name",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={
        "day_name": day_order,
        "balancing_authority": ba_order
    },
    title="Demand Distribution by Day of Week: Panel View",
    labels={
        "day_name": "Day of Week",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_dow_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_dow_panel.update_yaxes(matches=None)

fig_dow_panel.show()

#### Step 14F Summary

The day-of-week panel boxplots show how demand is distributed across weekdays and weekends for each California balancing authority. The boxplots are useful because they show the median, middle 50% range, spread, and outliers without needing additional reference lines.

CISO shows the highest demand levels across all days, with weekday demand generally appearing higher than weekend demand. This pattern is consistent with business, commercial, and industrial activity contributing more strongly during the workweek. IID and TIDC show smaller demand levels but still display visible weekday and weekend variation. LDWP shows a wider spread and several low-demand outliers, which matches earlier observations about very low reported demand during parts of the study period. BANC shows several high outliers, suggesting isolated demand spikes that should be interpreted carefully.

This chart supports the stress index design by showing that demand behavior varies not only by hour of day and date, but also by day of week. A strong stress feature should therefore preserve time-based context rather than treating all hours as equal.

#### Step 14G: Weekday vs Weekend Demand Comparison

The day-of-week boxplot provides detailed distribution by each calendar day. To simplify the pattern, this step groups days into weekday and weekend categories.

This comparison helps determine whether demand behavior changes meaningfully between workweek and weekend periods. A weekday-weekend feature may be useful later if demand patterns differ enough to support dashboard filtering or stress-index interpretation.

In [ ]:
# ============================================
# STEP 14G: WEEKDAY VS WEEKEND DEMAND COMPARISON
# ============================================

# Create weekday/weekend category
df_ca["day_type"] = df_ca["local_time_pacific"].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# Summarize demand by balancing authority and day type
weekday_weekend_summary = (
    df_ca
    .groupby(["balancing_authority", "day_type"], as_index=False)
    .agg(
        observations=("demand_mw", "count"),
        mean_demand_mw=("demand_mw", "mean"),
        median_demand_mw=("demand_mw", "median"),
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75)),
        max_demand_mw=("demand_mw", "max")
    )
    .round(2)
)

print("Weekday vs weekend demand summary:")
display(weekday_weekend_summary)

In [ ]:
# ============================================
# STEP 14G VISUAL: WEEKDAY VS WEEKEND DEMAND PANEL
# ============================================

fig_day_type_panel = px.box(
    df_ca,
    x="day_type",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={
        "day_type": ["Weekday", "Weekend"],
        "balancing_authority": ba_order
    },
    title="Weekday vs Weekend Demand Distribution by California Balancing Authority",
    labels={
        "day_type": "Day Type",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_day_type_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_day_type_panel.update_yaxes(matches=None)

fig_day_type_panel.show()

#### Step 14G Summary

The weekday versus weekend comparison shows that demand is generally higher on weekdays across all five California balancing authorities. This pattern is clearest for CISO, LDWP, and TIDC, where both mean and median demand are higher during the workweek than on weekends.

CISO has the largest weekday-weekend difference. Its average weekday demand is about **24,905 MW**, compared with about **22,904 MW** on weekends. LDWP also shows a meaningful difference, with average weekday demand of about **2,664 MW** compared with about **2,281 MW** on weekends. BANC, IID, and TIDC show smaller but still visible weekday demand increases.

This result suggests that workweek activity contributes to higher electricity demand across the California balancing authorities. It also supports the idea that `day_type` may be useful as a contextual feature or dashboard filter. However, compared with the strong hour-of-day patterns seen earlier, weekday versus weekend behavior appears to be a secondary time-based signal rather than the main driver of grid stress.

### Step 14 Summary: EDA Findings Before Stress Index Design

The exploratory analysis shows that California grid behavior cannot be summarized by demand alone. Demand varies by balancing authority, hour of day, date, and day type. CISO dominates in absolute megawatt scale, but smaller authorities such as BANC, IID, LDWP, and TIDC still show meaningful high-demand periods when measured relative to their own operating ranges.

The demand and forecast comparison showed that forecast error differs across authorities. CISO had the largest average forecast gap, while IID and TIDC had much smaller forecast errors. This suggests that forecast deviation may be an important operational signal, especially for larger or more variable authorities.

The demand and net generation comparison showed that supply balance matters. CISO, LDWP, BANC, and TIDC generally had demand above net generation, while IID often showed net generation above demand. The interchange analysis supported this pattern: CISO had the largest negative interchange values, suggesting frequent net import reliance, while IID showed positive average interchange.

The time-based visuals added further context. The median hourly profile with the IQR band showed that demand is strongly tied to hour of day, and the normalized heatmaps showed that high relative demand appears at different times for different authorities. The weekday versus weekend analysis showed that weekday demand is generally higher, but hour-of-day behavior appears to be the stronger time-based pattern.

These findings support the next phase of the project: building a grid stress index. The stress index should combine several operational signals, including relative demand level, forecast error, generation-demand gap, interchange pressure, and time-based context. Because California balancing authorities operate at very different scales, these features should be normalized within each balancing authority before comparing stress conditions statewide.

## Step 15: Feature Engineering: Grid Stress Index

The exploratory analysis showed that grid stress should not be measured by demand alone. High demand is important, but operational pressure also depends on whether demand was forecasted accurately, whether demand exceeds net generation, whether the authority is relying on imports, and whether the hour falls into a high-demand time window.

Because California balancing authorities operate at very different scales, the stress index is calculated using within-authority normalized features. This prevents CISO from automatically dominating the index simply because it has the largest raw demand values. Instead, each authority is evaluated relative to its own observed operating range.

The stress index combines four normalized signals:

| Component | Meaning |
|---|---|
| Demand pressure | How high demand is relative to the authority’s own demand history |
| Forecast error pressure | How large the absolute forecast error is relative to the authority’s own forecast-error history |
| Generation gap pressure | How high the demand-minus-generation gap is relative to the authority’s own gap history |
| Import pressure | How large net import reliance is relative to the authority’s own interchange history |

The resulting index ranges from low to high relative stress. Higher values indicate periods that may deserve closer review in the dashboard.

In [ ]:
# ============================================
# STEP 15: FEATURE ENGINEERING - GRID STRESS INDEX
# ============================================

# Create a working copy for feature engineering
df_stress = df_ca.copy()

# --------------------------------------------
# 1. Create base operational features
# --------------------------------------------

# Forecast error: positive means actual demand was higher than forecast
df_stress["forecast_error_mw"] = (
    df_stress["demand_mw"] - df_stress["demand_forecast_mw"]
)

# Absolute forecast error captures forecast miss size regardless of direction
df_stress["abs_forecast_error_mw"] = df_stress["forecast_error_mw"].abs()

# Generation gap: positive means demand is greater than net generation
df_stress["generation_gap_mw"] = (
    df_stress["demand_mw"] - df_stress["net_generation_mw"]
)

# Import pressure: negative interchange indicates net imports.
# Convert imports into positive values and exports into 0.
df_stress["import_pressure_mw"] = (
    -df_stress["total_interchange_mw"]
).clip(lower=0)

# Add time-context features
df_stress["hour_of_day"] = df_stress["local_time_pacific"].dt.hour
df_stress["day_name"] = df_stress["local_time_pacific"].dt.day_name()
df_stress["day_type"] = df_stress["local_time_pacific"].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# --------------------------------------------
# 2. Normalize stress components within each balancing authority
# --------------------------------------------

def percentile_rank_within_group(series):
    """
    Converts a numeric series into percentile ranks from 0 to 1.
    Higher values receive higher percentile scores.
    """
    return series.rank(pct=True)

# Demand pressure: high demand relative to the authority's own history
df_stress["demand_pressure_score"] = (
    df_stress
    .groupby("balancing_authority")["demand_mw"]
    .transform(percentile_rank_within_group)
)

# Forecast error pressure: high absolute forecast error relative to own history
df_stress["forecast_error_score"] = (
    df_stress
    .groupby("balancing_authority")["abs_forecast_error_mw"]
    .transform(percentile_rank_within_group)
)

# Generation gap pressure: high demand-minus-generation gap relative to own history
df_stress["generation_gap_score"] = (
    df_stress
    .groupby("balancing_authority")["generation_gap_mw"]
    .transform(percentile_rank_within_group)
)

# Import pressure: high net import reliance relative to own history
df_stress["import_pressure_score"] = (
    df_stress
    .groupby("balancing_authority")["import_pressure_mw"]
    .transform(percentile_rank_within_group)
)

# --------------------------------------------
# 3. Combine normalized components into stress index
# --------------------------------------------

# Equal-weight index for transparent interpretation
df_stress["grid_stress_index"] = (
    0.25 * df_stress["demand_pressure_score"] +
    0.25 * df_stress["forecast_error_score"] +
    0.25 * df_stress["generation_gap_score"] +
    0.25 * df_stress["import_pressure_score"]
)

# Convert to 0-100 scale for dashboard readability
df_stress["grid_stress_score"] = df_stress["grid_stress_index"] * 100

# --------------------------------------------
# 4. Assign stress categories
# --------------------------------------------

df_stress["stress_category"] = pd.cut(
    df_stress["grid_stress_score"],
    bins=[0, 50, 75, 90, 100],
    labels=[
        "Normal",
        "Elevated",
        "High",
        "Very High"
    ],
    include_lowest=True
)

print("Grid stress index created.")
print("Dataset shape:", df_stress.shape)

print("\nStress score summary:")
display(
    df_stress[
        [
            "grid_stress_score",
            "demand_pressure_score",
            "forecast_error_score",
            "generation_gap_score",
            "import_pressure_score"
        ]
    ].describe().round(2)
)

print("\nStress category counts:")
display(
    df_stress["stress_category"]
    .value_counts()
    .rename_axis("stress_category")
    .reset_index(name="count")
)

### Step 15 Summary

The grid stress index was created successfully for the California balancing-authority dataset. The final feature-engineered dataset contains **13,020 rows and 81 columns**.

The stress index combines four within-authority normalized signals: demand pressure, forecast error pressure, generation-demand gap pressure, and import pressure. Each component was scaled within its own balancing authority so that smaller authorities such as IID and TIDC can be compared fairly with larger authorities such as CISO.

The resulting stress score ranges from about **2.02** to **99.89**, with an average score of about **50.02**. This centered distribution is expected because the component scores were based on percentile ranks. Most observations fall into the Normal or Elevated categories, while a smaller number of records are classified as High or Very High.

There are fewer non-null stress scores than total rows because some records are missing demand forecast values. Since forecast error is one of the stress-index components, those records cannot receive a complete stress score unless the missing forecast values are imputed or the index is recalculated using available components only.

The category counts show that the stress index is useful for prioritization: **175 records** are classified as Very High stress, and **1,036 records** are classified as High stress. These periods should be reviewed next to confirm whether the stress index is identifying meaningful grid conditions.

In [ ]:
# ============================================
# STEP 15 CHECK: STRESS SUMMARY BY BALANCING AUTHORITY
# ============================================

stress_summary_by_ba = (
    df_stress
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("grid_stress_score", "count"),
        mean_stress_score=("grid_stress_score", "mean"),
        median_stress_score=("grid_stress_score", "median"),
        max_stress_score=("grid_stress_score", "max"),
        very_high_periods=("stress_category", lambda x: (x == "Very High").sum()),
        high_periods=("stress_category", lambda x: (x == "High").sum()),
        elevated_periods=("stress_category", lambda x: (x == "Elevated").sum())
    )
    .round(2)
)

print("Stress summary by balancing authority:")
display(stress_summary_by_ba)

In [ ]:
# ============================================
# STEP 15 CHECK: REVIEW HIGHEST STRESS PERIODS
# ============================================

top_stress_periods = (
    df_stress
    .sort_values("grid_stress_score", ascending=False)
    [
        [
            "balancing_authority",
            "local_time_pacific",
            "hour_of_day",
            "day_name",
            "day_type",
            "demand_mw",
            "demand_forecast_mw",
            "forecast_error_mw",
            "net_generation_mw",
            "generation_gap_mw",
            "total_interchange_mw",
            "import_pressure_mw",
            "grid_stress_score",
            "stress_category"
        ]
    ]
    .head(20)
)

print("Top 20 highest stress periods:")
display(top_stress_periods)

### Step 15 Validation Summary

The stress-index validation shows that the engineered score behaves as intended. The mean stress scores are close to 50 for all five balancing authorities, which is expected because the main stress components were normalized within each authority using percentile ranks. This confirms that the index avoids letting CISO dominate only because of its larger raw demand scale.

The highest stress periods are concentrated in records where multiple pressure signals occur together. For example, the top BANC periods combine unusually high demand, large forecast errors, large generation-demand gaps, and large negative interchange values. IID, TIDC, and LDWP also appear in the highest stress records when their own demand, generation gap, and import pressure are high relative to their normal operating ranges.

The category counts by balancing authority show that the stress index identifies authority-specific risk patterns. CISO has the largest raw demand but only one Very High stress period, while LDWP, BANC, TIDC, and IID show more Very High periods relative to their own operating behavior. This confirms that the index is not simply measuring grid size. It is measuring relative operational pressure.

This validation supports using the grid stress score as the main engineered feature for dashboard development and ranking higher-priority operating periods.

In [ ]:
# ============================================
# STEP 15 FINAL CLEANUP: KEEP COMPLETE STRESS SCORE RECORDS
# ============================================

df_stress_complete = df_stress.dropna(subset=["grid_stress_score"]).copy()

print("Original stress dataset rows:", len(df_stress))
print("Complete stress-score rows:", len(df_stress_complete))
print("Rows removed due to missing stress score:", len(df_stress) - len(df_stress_complete))

### Step 15 Cleanup Summary

The complete stress-score dataset contains **12,735 records** out of the original **13,020 California records**. A total of **285 rows** were removed because they did not have a complete stress score.

The missing stress scores are primarily caused by missing demand forecast values. Since forecast error is one of the four stress-index components, those records cannot receive a fully comparable stress score without imputation or a modified scoring rule.

For this version of the project, the dashboard-ready stress dataset uses only complete stress-score records. This keeps the index transparent and ensures that every exported record is scored using the same four components: demand pressure, forecast error pressure, generation-gap pressure, and import pressure.

## Step 16: Visualize Grid Stress Index Results

After constructing and validating the grid stress index, the next step is to visualize the stress score and stress categories. These visuals help confirm whether the engineered feature behaves logically over time and whether high-stress periods are distributed across balancing authorities in a meaningful way.

This step uses the complete stress-score dataset, `df_stress_complete`, so every record shown in the visuals has all four stress-index components available: demand pressure, forecast error pressure, generation-gap pressure, and import pressure.

### Step 16A: Stress Score Over Time by Balancing Authority

The first stress-index visualization plots the grid stress score over time for each California balancing authority. A panel view is used because the authorities have different operating patterns, and separate panels make it easier to see stress movement within each authority.

This view helps identify whether stress periods are isolated spikes, repeated clusters, or broader sustained patterns.

In [ ]:
# ============================================
# STEP 16A: STRESS SCORE OVER TIME BY BALANCING AUTHORITY
# WITH THRESHOLD LINES IN LEGEND
# ============================================

import plotly.graph_objects as go

df_stress_complete_sorted = df_stress_complete.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

fig_stress_time = px.line(
    df_stress_complete_sorted,
    x="local_time_pacific",
    y="grid_stress_score",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Grid Stress Score Over Time by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

# Clean facet labels
fig_stress_time.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Keep same y-axis range for stress score because it is normalized
fig_stress_time.update_yaxes(range=[0, 100])

# Add threshold lines to each panel
fig_stress_time.add_hline(
    y=50,
    line_dash="dot",
    line_color="gray",
    line_width=1.5
)

fig_stress_time.add_hline(
    y=75,
    line_dash="dash",
    line_color="orange",
    line_width=1.5
)

fig_stress_time.add_hline(
    y=90,
    line_dash="dash",
    line_color="red",
    line_width=1.5
)

# Add dummy traces so threshold lines appear in the legend
# These traces are not used for data plotting. They only create legend entries.
fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="gray", dash="dot", width=1.5),
        name="Elevated threshold: 50",
        showlegend=True
    )
)

fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="orange", dash="dash", width=1.5),
        name="High threshold: 75",
        showlegend=True
    )
)

fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="red", dash="dash", width=1.5),
        name="Very High threshold: 90",
        showlegend=True
    )
)

# Improve legend placement
fig_stress_time.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_stress_time.show()

### Step 16A Summary

The stress score time-series chart shows how relative grid stress changes over time for each California balancing authority. Because the stress score is normalized within each authority, the panels can be compared on the same 0–100 scale without allowing CISO’s larger raw demand to dominate the visual.

The reference lines mark the stress thresholds used in the index: **50** for Elevated, **75** for High, and **90** for Very High. Periods above these thresholds indicate hours where multiple stress signals occur together, including relative demand pressure, forecast error, generation-demand gap, and import pressure.

The chart shows that stress periods occur across all five California authorities, but the timing and pattern differ by authority. BANC and LDWP show several sharp stress spikes, while IID and TIDC show more visible elevated periods later in the study window. CISO has fewer Very High periods even though it has the largest raw demand, which confirms that the index is measuring relative operating pressure rather than system size alone.

This visualization supports dashboard development because it provides a time-based monitoring view of stress conditions across California balancing authorities.

### Step 16B: Review Priority Classification

The grid stress score is a continuous metric from 0 to 100. Continuous scores are useful for ranking records, but dashboards also need simple categories that help users filter and prioritize periods for review.

To support dashboard interpretation, the stress score is converted into a review-priority label. This label is separate from the analytical stress category. The stress category describes the full score range, while the review-priority label simplifies the result into action-oriented tiers.

| Review Priority | Stress Score Range | Interpretation |
|---|---:|---|
| High Review Priority | `>= HIGH_REVIEW_THRESHOLD` | Periods where multiple stress signals are unusually high and deserve immediate review |
| Medium Review Priority | `>= MEDIUM_REVIEW_THRESHOLD` and `< HIGH_REVIEW_THRESHOLD` | Periods with elevated stress conditions that should be monitored in trend context |
| Low Review Priority | `< MEDIUM_REVIEW_THRESHOLD` | Lower-priority periods within the observed dataset window |

The threshold values are defined as named constants so the classification logic can be adjusted without rewriting the function body.

In [ ]:
# ============================================
# STEP 16B: REVIEW PRIORITY CLASSIFICATION
# ============================================

# Named constants for dashboard review-priority classification
MEDIUM_REVIEW_THRESHOLD = 75
HIGH_REVIEW_THRESHOLD = 90

def assign_review_priority(score):
    """
    Converts a continuous grid stress score into a dashboard-friendly
    review priority label.
    """
    if pd.isna(score):
        return "Missing Score"
    elif score >= HIGH_REVIEW_THRESHOLD:
        return "High Review Priority"
    elif score >= MEDIUM_REVIEW_THRESHOLD:
        return "Medium Review Priority"
    else:
        return "Low Review Priority"

# Apply review-priority classification to both stress datasets
df_stress["review_priority"] = df_stress["grid_stress_score"].apply(
    assign_review_priority
)

df_stress_complete["review_priority"] = df_stress_complete["grid_stress_score"].apply(
    assign_review_priority
)

print("Review priority classification created.")

print("\nReview priority counts:")
display(
    df_stress_complete["review_priority"]
    .value_counts()
    .rename_axis("review_priority")
    .reset_index(name="count")
)

print("\nReview priority counts by balancing authority:")
display(
    df_stress_complete
    .groupby(["balancing_authority", "review_priority"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

### Step 16B Summary

The review-priority classification was created successfully from the continuous grid stress score. The complete scored dataset contains **12,735 records**. Most records are classified as **Low Review Priority**, with **11,524 records** below the medium threshold of 75. There are **1,036 Medium Review Priority** records and **175 High Review Priority** records.

The by-authority summary shows that high-priority records are distributed across the California balancing authorities rather than being dominated by CISO. LDWP has the largest number of High Review Priority records with **69**, followed by BANC with **43**, TIDC with **35**, IID with **27**, and CISO with **1**.

This distribution supports the design of the grid stress score. Since the score is normalized within each balancing authority and combines demand pressure, forecast error pressure, generation-gap pressure, and import pressure, smaller authorities can appear in the high-priority review queue when they experience unusual pressure relative to their own operating range.

The review-priority label is now ready to support dashboard filtering, ranked review tables, and summary visuals.

### Step 16C: Review Priority Counts by Balancing Authority

This step visualizes the review-priority counts by balancing authority. The chart translates the stress score into a practical dashboard view, showing how many records from each authority fall into Low, Medium, and High Review Priority categories.

This makes the priority distribution easier to compare across authorities and supports dashboard-level filtering and monitoring.

In [ ]:
# ============================================
# STEP 16C: REVIEW PRIORITY COUNTS BY BALANCING AUTHORITY
# ============================================

review_priority_counts = (
    df_stress_complete
    .groupby(["balancing_authority", "review_priority"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

print("Review priority counts by balancing authority:")
display(review_priority_counts)

fig_review_priority = px.bar(
    review_priority_counts,
    x="balancing_authority",
    y="count",
    color="review_priority",
    barmode="group",
    category_orders={
        "balancing_authority": ba_order,
        "review_priority": [
            "Low Review Priority",
            "Medium Review Priority",
            "High Review Priority"
        ]
    },
    title="Review Priority Counts by California Balancing Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "count": "Record Count",
        "review_priority": "Review Priority"
    }
)

fig_review_priority.show()

### Step 16C Summary

The review-priority count chart shows that most scored records fall into the Low Review Priority category across all five California balancing authorities. This is expected because only scores at or above 75 are assigned to Medium or High Review Priority.

LDWP has the largest number of High Review Priority records with **69**, followed by BANC with **43**, TIDC with **35**, IID with **27**, and CISO with **1**. This pattern is important because it shows that high-priority periods are not determined by raw demand scale alone. CISO has the largest demand in megawatts, but it does not dominate the high-priority count.

The chart supports the dashboard design because it gives users a compact way to compare review-priority distribution by authority. It also confirms that the stress score is functioning as a relative review signal, identifying unusual pressure within each authority’s own operating range.

### Step 16C2: Review Priority Percentage by Balancing Authority

The count chart shows the number of records in each review-priority tier, but the authorities do not all have exactly the same number of complete scored observations. A percentage view makes the comparison fairer by showing the share of each authority’s records assigned to each priority tier.

In [ ]:
# ============================================
# STEP 16C2: REVIEW PRIORITY PERCENTAGE BY BALANCING AUTHORITY
# ============================================

review_priority_pct = review_priority_counts.copy()

review_priority_pct["total_by_authority"] = (
    review_priority_pct
    .groupby("balancing_authority")["count"]
    .transform("sum")
)

review_priority_pct["percentage"] = (
    review_priority_pct["count"] / review_priority_pct["total_by_authority"] * 100
).round(2)

print("Review priority percentage by balancing authority:")
display(review_priority_pct)

fig_review_priority_pct = px.bar(
    review_priority_pct,
    x="balancing_authority",
    y="percentage",
    color="review_priority",
    barmode="group",
    category_orders={
        "balancing_authority": ba_order,
        "review_priority": [
            "Low Review Priority",
            "Medium Review Priority",
            "High Review Priority"
        ]
    },
    title="Review Priority Share by California Balancing Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "percentage": "Share of Scored Records (%)",
        "review_priority": "Review Priority"
    },
    text="percentage"
)

fig_review_priority_pct.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig_review_priority_pct.update_yaxes(range=[0, 100])

fig_review_priority_pct.show()

### Step 16C2 Summary

The review-priority percentage chart shows the share of each balancing authority’s scored records that fall into Low, Medium, and High Review Priority categories. This percentage view gives a fairer comparison than raw counts because the authorities do not all have the exact same number of complete scored observations.

Most records remain Low Review Priority for every authority. CISO has the highest Low Review Priority share at **94.50%**, with only **0.04%** of its scored records classified as High Review Priority. LDWP has the highest High Review Priority share at **2.91%** and the highest Medium Review Priority share at **12.47%**. BANC, IID, and TIDC fall between these patterns, with High Review Priority shares of **1.67%**, **1.03%**, and **1.34%**, respectively.

This chart strengthens the dashboard interpretation because it shows review-priority intensity as a percentage of each authority’s own scored records, not just as raw counts. It also confirms that the grid stress score is functioning as a relative signal rather than a raw demand ranking.

### Step 16D: Identify Top Periods for Review

A decision-maker often needs a short, ranked list rather than a full dataset. This step produces a compact review table of the highest-scoring periods across all California balancing authorities.

The table includes demand, forecast error, net generation, generation gap, interchange, import pressure, grid stress score, and review priority. These fields provide operational context so that the top-ranked periods can be interpreted as more than simple high-demand hours.

In [ ]:
# ============================================
# STEP 16D: IDENTIFY TOP PERIODS FOR REVIEW
# ============================================

TOP_N_REVIEW_PERIODS = 20

top_review_periods = (
    df_stress_complete
    .sort_values("grid_stress_score", ascending=False)
    .loc[:, [
        "local_time_pacific",
        "balancing_authority",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "grid_stress_score",
        "review_priority"
    ]]
    .head(TOP_N_REVIEW_PERIODS)
    .reset_index(drop=True)
)

top_review_periods.index += 1
top_review_periods.index.name = "Rank"

print(f"Top {TOP_N_REVIEW_PERIODS} periods for review:")
display(top_review_periods)

### Step 16D Summary

The top review periods table identifies the 20 highest-scoring grid stress periods in the complete California stress dataset. All listed records are classified as High Review Priority, with grid stress scores ranging from about **97.54** to **99.89**.

The table confirms that the stress score is not simply ranking raw demand. CISO, the largest authority by megawatt demand, does not dominate the top 20 list. Instead, BANC, IID, TIDC, and LDWP appear because their stress scores are calculated relative to their own demand, forecast error, generation gap, and import pressure patterns.

Several of the highest-ranked BANC periods show a clear combination of high demand, large forecast error, large generation-demand gap, and large negative interchange. This explains why they rise to the top of the review queue. IID and TIDC also appear frequently in the top 20 when their demand and import pressure are high relative to their normal operating range.

This table is useful for dashboard development because it provides a compact review queue. It reduces the full scored dataset to a short ranked list of periods that deserve closer review, while still preserving enough operational context to explain why each record was prioritized.

### Step 16E: Visualize Top Review Periods

The top review periods are visualized as a ranked bar chart. This makes the review queue easier to scan and helps show which balancing authorities appear among the highest stress scores.

The bar chart complements the detailed table by providing a quick visual summary of the highest-priority periods.

In [ ]:
# ============================================
# STEP 16E: VISUALIZE TOP REVIEW PERIODS
# ============================================

top_review_periods_plot = top_review_periods.reset_index()

fig_top_review = px.bar(
    top_review_periods_plot.sort_values("grid_stress_score", ascending=False),
    x="Rank",
    y="grid_stress_score",
    color="balancing_authority",
    hover_data=[
        "local_time_pacific",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "review_priority"
    ],
    title=f"Top {TOP_N_REVIEW_PERIODS} Grid Stress Periods for Review",
    labels={
        "Rank": "Review Rank",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    text="grid_stress_score",
    width=1200,
    height=550
)

fig_top_review.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig_top_review.update_yaxes(range=[0, 105])

fig_top_review.show()

### Step 16E Summary

The ranked bar chart shows the top 20 grid stress periods in the complete California stress dataset. All of the displayed records have grid stress scores above **97.5**, meaning they are very close to the top of the 0-100 stress scale.

The chart shows that BANC dominates the highest ranks, especially the first several positions. IID, TIDC, and LDWP also appear in the top 20. CISO does not appear in this top review chart, even though it has the largest raw demand in megawatts. This confirms that the grid stress score is not simply a raw demand ranking.

This visualization supports the project’s main design principle: review priority should be based on relative operational pressure within each balancing authority. By using normalized stress components, the review queue can surface high-pressure periods from smaller authorities that would be hidden in a raw megawatt ranking.

In [ ]:
# ============================================
# STEP 16E2: ZOOMED VIEW OF TOP REVIEW PERIODS
# ============================================

fig_top_review_zoom = px.bar(
    top_review_periods_plot.sort_values("grid_stress_score", ascending=False),
    x="Rank",
    y="grid_stress_score",
    color="balancing_authority",
    hover_data=[
        "local_time_pacific",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "review_priority"
    ],
    title=f"Top {TOP_N_REVIEW_PERIODS} Grid Stress Periods for Review: Zoomed View",
    labels={
        "Rank": "Review Rank",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    text="grid_stress_score",
    width=1200,
    height=550
)

fig_top_review_zoom.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False
)

fig_top_review_zoom.update_yaxes(
    range=[95, 100.35]   # give extra headroom above tallest bar
)

fig_top_review_zoom.update_layout(
    margin=dict(t=100, r=40, l=60, b=60),   # more room at the top
    uniformtext_minsize=10,
    uniformtext_mode="show"
)

fig_top_review_zoom.show()

### Step 16E2 Summary

The zoomed top review chart provides a clearer view of the highest-ranked grid stress periods. All top 20 records have stress scores above **97.5**, so the zoomed y-axis helps show the small differences between records that would be harder to see on a full 0–100 scale.

BANC occupies the highest-ranked positions, followed by IID, TIDC, and LDWP. CISO does not appear in the top 20, even though it has the largest raw demand. This confirms that the grid stress score is not a raw megawatt ranking. Instead, it ranks periods based on relative pressure within each balancing authority.

This chart is useful for dashboard storytelling because it turns the stress index into a practical review queue. A user can quickly identify the highest-priority periods and then use the supporting table to inspect demand, forecast error, generation gap, and interchange conditions for each ranked hour.

## Step 17: Key Findings Summary

This section calculates the main project findings directly from the complete stress-score dataset. These values can be used in the project README, dashboard description, portfolio write-up, or executive summary.

The summary includes the total number of California records, the number of complete scored records, review-priority counts, peak demand, average grid stress score, maximum grid stress score, and the balancing authority with the highest number of high-priority review periods.

Calculating these values directly in the notebook creates a reproducible audit trail. If the dataset or stress-index logic changes later, the summary can be regenerated automatically instead of being updated manually.

In [ ]:
# ============================================
# STEP 17: KEY FINDINGS SUMMARY
# ============================================

# Total records in the California-only dataset
total_california_records = len(df_ca)

# Total records with complete grid stress scores
total_scored_records = len(df_stress_complete)

# Review-priority counts
priority_counts = df_stress_complete["review_priority"].value_counts()

high_count = priority_counts.get("High Review Priority", 0)
medium_count = priority_counts.get("Medium Review Priority", 0)
low_count = priority_counts.get("Low Review Priority", 0)

# Review-priority percentages
high_pct = round((high_count / total_scored_records) * 100, 2)
medium_pct = round((medium_count / total_scored_records) * 100, 2)
low_pct = round((low_count / total_scored_records) * 100, 2)

# Peak demand record
peak_row = df_stress_complete.loc[df_stress_complete["demand_mw"].idxmax()]
peak_ba = peak_row["balancing_authority"]
peak_demand = peak_row["demand_mw"]
peak_time = peak_row["local_time_pacific"]

# Stress score summary
avg_stress_score = round(df_stress_complete["grid_stress_score"].mean(), 2)
max_stress_score = round(df_stress_complete["grid_stress_score"].max(), 2)

# Balancing authority with the most high-priority periods
high_priority_by_ba = (
    df_stress_complete[
        df_stress_complete["review_priority"] == "High Review Priority"
    ]
    .groupby("balancing_authority")
    .size()
    .sort_values(ascending=False)
)

top_high_priority_ba = high_priority_by_ba.index[0]
top_high_priority_count = high_priority_by_ba.iloc[0]

# Print key findings
print("=== Key Findings: California Grid Dataset (Jan-Apr 2026) ===")
print(f"Total California records:                  {total_california_records:,}")
print(f"Complete scored records:                   {total_scored_records:,}")
print(f"Records excluded from scoring:             {total_california_records - total_scored_records:,}")
print()
print(f"High Review Priority periods:              {high_count:,}  ({high_pct:.2f}% of scored records)")
print(f"Medium Review Priority periods:            {medium_count:,}  ({medium_pct:.2f}% of scored records)")
print(f"Low Review Priority periods:               {low_count:,}  ({low_pct:.2f}% of scored records)")
print()
print(f"Peak demand single hour:                   {peak_demand:,.0f} MW - {peak_ba}")
print(f"Peak demand timestamp:                     {peak_time}")
print()
print(f"Average grid stress score:                 {avg_stress_score}")
print(f"Maximum grid stress score:                 {max_stress_score}")
print()
print(
    "Authority with most High Review Priority periods: "
    f"{top_high_priority_ba} ({top_high_priority_count:,})"
)

### Step 17A: Visualize Review Priority Composition

After classifying each scored record into Low, Medium, or High Review Priority, the next step is to show how the scored California grid records are distributed across these categories.

A donut chart is used here because the analytical question is compositional: What share of scored records fall into each review-priority tier? This chart provides a compact summary and makes the small High Review Priority share easier to emphasize visually than a stretched bar chart.

The chart keeps the project’s semantic color scheme:
- Green = Low Review Priority
- Yellow = Medium Review Priority
- Red = High Review Priority

The center annotation highlights the High Review Priority share because that group represents the smallest but most operationally important set of records.

In [ ]:
import plotly.graph_objects as go

# ============================================
# STEP 17A: REVIEW PRIORITY COMPOSITION
# ============================================

# Ensure priority order is consistent
priority_order = [
    "Low Review Priority",
    "Medium Review Priority",
    "High Review Priority"
]

priority_color_map = {
    "Low Review Priority": "#2ca02c",      # green
    "Medium Review Priority": "#e6c84f",   # yellow-gold
    "High Review Priority": "#d62728"      # red
}

priority_share_stack = priority_share.copy()

priority_share_stack["review_priority"] = pd.Categorical(
    priority_share_stack["review_priority"],
    categories=priority_order,
    ordered=True
)

priority_share_stack = priority_share_stack.sort_values("review_priority")

priority_colors = [priority_color_map[p] for p in priority_share_stack["review_priority"]]

# High-priority percentage for center KPI
high_val = priority_share_stack.loc[
    priority_share_stack["review_priority"] == "High Review Priority",
    "percentage"
].iloc[0]

fig_priority_donut = go.Figure()

fig_priority_donut.add_trace(go.Pie(
    labels=priority_share_stack["review_priority"],
    values=priority_share_stack["percentage"],
    hole=0.62,
    sort=False,
    direction="clockwise",
    marker=dict(
        colors=priority_colors,
        line=dict(color="white", width=2)
    ),
    textinfo="percent",
    textposition="outside",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Share: %{value:.2f}%<extra></extra>"
    ),
    pull=[0, 0, 0.12]   # emphasize High Review Priority
))

# Center annotation
fig_priority_donut.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",
    yref="paper",
    text=f"<b>{high_val:.2f}%</b><br>High Review<br>Priority",
    showarrow=False,
    font=dict(size=16, color="#d62728"),
    align="center"
)

fig_priority_donut.update_layout(
    title=dict(
        text="Operational Risk: Review Priority Composition",
        x=0.5,
        xanchor="center"
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.08,
        xanchor="center",
        x=0.5,
        title="Review Priority"
    ),
    width=700,
    height=560,
    margin=dict(t=90, b=110, l=60, r=60)
)

fig_priority_donut.show()

### Step 17A Summary

The donut chart shows that the California grid dataset is dominated by Low Review Priority periods, which account for 90.49% of scored records. Medium Review Priority periods account for 8.14%, while High Review Priority periods account for only 1.37%.

This is an important result. The review-priority system sharply narrows the dataset from thousands of scored hourly records to a very small subset of periods that deserve immediate operational attention. In other words, the grid stress framework is not merely descriptive; it functions as a triage tool.

The highlighted High Review Priority segment is small by design. That is desirable, because the purpose of the classification is to identify a limited number of exceptional periods rather than label a large share of records as critical.

### Step 17B: High-Priority Risk Concentration by Authority

After reviewing the overall priority composition, this step focuses only on High Review Priority periods. The goal is to identify which California balancing authorities contributed the largest number of high-priority records.

A horizontal bar chart is used because it makes the ranking easy to read. Sorting the authorities by count places the highest concentration at the top of the chart, making the main finding immediately visible.

In [ ]:
# ============================================
# STEP 17B: HIGH-PRIORITY RISK CONCENTRATION BY AUTHORITY
# ============================================

# Count High Review Priority periods by balancing authority
high_priority_counts = (
    df_stress_complete[
        df_stress_complete["review_priority"] == "High Review Priority"
    ]
    .groupby("balancing_authority", as_index=False)
    .size()
    .rename(columns={"size": "high_priority_count"})
)

# Sort so the largest value appears at the top of the horizontal chart
high_priority_counts = high_priority_counts.sort_values(
    "high_priority_count",
    ascending=True
)

fig_high_priority = px.bar(
    high_priority_counts,
    y="balancing_authority",
    x="high_priority_count",
    orientation="h",
    title="High-Priority Risk Concentration by Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "high_priority_count": "High Review Priority Periods"
    },
    color="high_priority_count",
    color_continuous_scale="Reds",
    text="high_priority_count",
    width=850,
    height=450
)

fig_high_priority.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker_line_color="rgb(8,48,107)",
    marker_line_width=1.5,
    opacity=0.85,
    width=0.6
)

fig_high_priority.update_xaxes(
    showgrid=True,
    gridcolor="LightGray",
    range=[0, high_priority_counts["high_priority_count"].max() * 1.18]
)

fig_high_priority.update_layout(
    plot_bgcolor="white",
    yaxis=dict(title=""),
    showlegend=False,
    coloraxis_showscale=False,
    margin=dict(l=100, r=60, t=80, b=50)
)

fig_high_priority.show(config={"responsive": False})

### Step 17B Summary

The high-priority risk concentration chart shows how High Review Priority periods are distributed across California balancing authorities. LDWP has the largest number of High Review Priority periods with 69 records, followed by BANC with 43, TIDC with 35, IID with 27, and CISO with 1.

This result reinforces the main purpose of the grid stress score. CISO has the largest raw demand in megawatts, but it does not dominate the high-priority review queue. The score identifies relative operational pressure within each authority by combining demand pressure, forecast error pressure, generation-demand gap pressure, and import pressure.

This chart is useful for executive reporting because it quickly shows where the highest-priority review burden is concentrated during the study period.

### Step 17 Summary

The key findings summary confirms that the California-only dataset contains **13,020 records**, of which **12,735 records** have complete grid stress scores. A total of **285 records** were excluded from scoring because at least one required stress-index component was missing, primarily due to missing demand forecast values.

Most scored records fall into the Low Review Priority category. There are **11,524 Low Review Priority periods**, representing **90.49%** of the scored dataset. There are **1,036 Medium Review Priority periods**, representing **8.14%**, and **175 High Review Priority periods**, representing **1.37%**. This distribution is useful because it creates a focused review queue instead of treating all hourly records as equally important.

The highest raw demand hour occurred in CISO at **35,596 MW** on **March 20, 2026 at 6:00 PM Pacific Time**. However, the authority with the most High Review Priority periods is **LDWP**, with **69 high-priority periods**. This contrast reinforces the value of the grid stress score: the project is not simply ranking the largest raw demand values, but identifying relative operational pressure within each authority.

The average grid stress score is **50.02**, and the maximum score is **99.89**. These values are consistent with the percentile-based scoring design and confirm that the final stress score can support dashboard ranking, filtering, and review-priority analysis.

## Step 18: Save Core Analysis Output Tables

This step saves the two core analysis output tables from the completed grid stress workflow.

The first file is the complete California grid stress dataset. It preserves the cleaned California records with engineered fields such as forecast error, generation gap, import pressure, grid stress score, stress category, and review priority. This file is the main processed analytical dataset.

The second file is the ranked top review periods table. It provides a compact review queue of the highest-scoring grid stress periods, including the supporting operational fields needed to explain why each period was prioritized.

Only these two files are saved in this step because they represent the main analysis outputs. The dashboard-specific data model is exported separately in the next phase, where purpose-built files are created for Tableau and Power BI.

In [ ]:
# ============================================
# STEP 18: SAVE FINAL OUTPUT TABLES
# ============================================

# Save ranked review table to outputs/
top_review_periods.to_csv(
    f"{OUTPUT_DIR}/top_review_periods_california.csv",
    index=True
)

# Save complete scored California grid dataset to data/processed/
df_stress_complete.to_csv(
    f"{PROCESSED_DIR}/cleaned_california_grid_stress_data.csv",
    index=False
)

print("Saved final California grid output tables successfully.")
print(f"Saved: {OUTPUT_DIR}/top_review_periods_california.csv")
print(f"Saved: {PROCESSED_DIR}/cleaned_california_grid_stress_data.csv")

print("\nExported table shapes:")
print("Top review periods:", top_review_periods.shape)
print("Complete stress dataset:", df_stress_complete.shape)

### Step 18 Summary

The core analysis output tables were saved successfully.

The ranked review table, `top_review_periods_california.csv`, contains **20 rows and 14 columns**. This file provides a compact review queue of the highest-scoring grid stress periods, including supporting operational context such as demand, forecast error, generation gap, interchange, grid stress score, and review priority.

The complete stress dataset, `cleaned_california_grid_stress_data.csv`, contains **12,735 rows and 82 columns**. This file preserves the California-only records with complete grid stress scores and engineered fields. It is the main processed analytical dataset produced by the notebook.

These two files represent the core analysis outputs. The next phase will create separate dashboard-ready files that are smaller, purpose-built, and easier to connect to Tableau or Power BI.

## Phase 4: Dashboard Data Model Export

The core analysis outputs saved in Step 18 preserve the completed stress-score dataset and the ranked review table. This phase creates a separate dashboard data model for Tableau and the planned Power BI implementation.

Instead of connecting the dashboard to one wide analytical file, this phase exports four purpose-built CSV files:

| File | Purpose |
|---|---|
| `california_grid_dashboard_ready.csv` | Hourly detail table for time-series charts, filters, review tables, and record-level drilldowns |
| `california_grid_monthly_summary.csv` | Month-level summary table for trend analysis by balancing authority |
| `california_grid_hourly_risk_summary.csv` | Hour-of-day summary table for daily-cycle and risk-profile visuals |
| `california_grid_kpi_summary.csv` | One-row-per-authority table for dashboard KPI cards |

These files are designed to make dashboard building cleaner, faster, and easier to explain. They can be regenerated by running the notebook from top to bottom.

The dashboard data model exports four purpose-built CSV files for Tableau and the planned Power BI implementation. The exact row and column counts are printed after export so the notebook documents the final data model directly from the generated files.

In [ ]:
# ============================================
# PHASE 4: DASHBOARD DATA MODEL EXPORT
# ============================================

# Work from complete scored records only
dashboard_source = df_stress_complete.copy()

# Add reusable time fields for dashboard filtering and aggregation
dashboard_source["year_month"] = dashboard_source["local_time_pacific"].dt.strftime("%Y-%m")
dashboard_source["local_date"] = dashboard_source["local_time_pacific"].dt.date
dashboard_source["hour_of_day"] = dashboard_source["local_time_pacific"].dt.hour

# ============================================
# FILE 1: DASHBOARD-READY HOURLY DETAIL
# ============================================

DASHBOARD_COLS = [
    "balancing_authority",
    "local_time_pacific",
    "local_date",
    "year_month",
    "hour_of_day",
    "day_name",
    "day_type",
    "demand_mw",
    "demand_forecast_mw",
    "forecast_error_mw",
    "abs_forecast_error_mw",
    "net_generation_mw",
    "generation_gap_mw",
    "total_interchange_mw",
    "import_pressure_mw",
    "grid_stress_score",
    "stress_category",
    "review_priority"
]

df_dashboard = dashboard_source[DASHBOARD_COLS].copy()

dashboard_path = f"{PROCESSED_DIR}/california_grid_dashboard_ready.csv"
df_dashboard.to_csv(dashboard_path, index=False)

print(f"Saved: {dashboard_path}  |  shape: {df_dashboard.shape}")


# ============================================
# FILE 2: MONTHLY SUMMARY
# ============================================

monthly_summary = (
    dashboard_source
    .groupby(["balancing_authority", "year_month"], as_index=False)
    .agg(
        avg_demand_mw=("demand_mw", "mean"),
        max_demand_mw=("demand_mw", "max"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_scored_hours=("grid_stress_score", "count"),
        high_priority_hours=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    monthly_summary[col] = monthly_summary[col].round(2)

monthly_path = f"{PROCESSED_DIR}/california_grid_monthly_summary.csv"
monthly_summary.to_csv(monthly_path, index=False)

print(f"Saved: {monthly_path}  |  shape: {monthly_summary.shape}")


# ============================================
# FILE 3: HOURLY RISK SUMMARY
# ============================================

hourly_risk = (
    dashboard_source
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        avg_demand_mw=("demand_mw", "mean"),
        max_demand_mw=("demand_mw", "max"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_observations=("grid_stress_score", "count"),
        high_priority_count=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_count=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_count=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    hourly_risk[col] = hourly_risk[col].round(2)

hourly_path = f"{PROCESSED_DIR}/california_grid_hourly_risk_summary.csv"
hourly_risk.to_csv(hourly_path, index=False)

print(f"Saved: {hourly_path}  |  shape: {hourly_risk.shape}")


# ============================================
# FILE 4: KPI SUMMARY
# ============================================

kpi_summary = (
    dashboard_source
    .groupby("balancing_authority", as_index=False)
    .agg(
        peak_demand_mw=("demand_mw", "max"),
        avg_demand_mw=("demand_mw", "mean"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_scored_hours=("grid_stress_score", "count"),
        date_range_start=("local_time_pacific", "min"),
        date_range_end=("local_time_pacific", "max"),
        high_priority_hours=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    kpi_summary[col] = kpi_summary[col].round(2)

kpi_summary["pct_high_priority"] = (
    kpi_summary["high_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["pct_medium_priority"] = (
    kpi_summary["medium_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["pct_low_priority"] = (
    kpi_summary["low_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["date_range_start"] = kpi_summary["date_range_start"].astype(str)
kpi_summary["date_range_end"] = kpi_summary["date_range_end"].astype(str)

kpi_path = f"{PROCESSED_DIR}/california_grid_kpi_summary.csv"
kpi_summary.to_csv(kpi_path, index=False)

print(f"Saved: {kpi_path}  |  shape: {kpi_summary.shape}")

print()
print("All four dashboard-ready files exported successfully.")
print("Note: These files are excluded from version control by .gitignore.")

### Phase 4 Export Summary

All four dashboard-ready files were exported successfully. These files are purpose-built for Tableau and the planned Power BI implementation.

| File | Rows | Columns | Role |
|---|---:|---:|---|
| `california_grid_dashboard_ready.csv` | 12,735 | 18 | Hourly detail table for time-series charts, filters, review queues, and record-level drilldowns |
| `california_grid_monthly_summary.csv` | 20 | 13 | Month-level trend table by balancing authority |
| `california_grid_hourly_risk_summary.csv` | 120 | 13 | Hour-of-day demand and stress profile for daily-cycle analysis |
| `california_grid_kpi_summary.csv` | 5 | 17 | KPI card table with one row per balancing authority |

The hourly detail file contains only complete scored records, so every exported row has the required stress-index components, grid stress score, stress category, and review priority. The monthly, hourly, and KPI summary files are derived from the same complete scored dataset, which keeps the dashboard model consistent.

These exports separate the dashboard layer from the wider analytical dataset. This makes Tableau and Power BI development cleaner because each file has a specific purpose: hourly detail, monthly trends, hour-of-day risk patterns, and KPI cards.

### Note: Published Dashboard and Notebook Versions

The export summary above accurately describes what the **current notebook code** produces: a multi-signal **Grid Stress Score** combining four normalized signals (demand pressure, forecast error pressure, generation gap pressure, and import pressure). This yields 12,735 complete records and an 18-column export.

The published **Tableau dashboard** was built on an earlier version of the pipeline using a simpler **Stress Index** defined as `(demand_mw / peak_demand_mw) × 100`. That version scored all 13,020 California hourly records and exported a 9-column dashboard-ready file.

**Relationship between the two versions:**

| Version | Records | Columns | Metric | Status |
|---|---:|---:|---|---|
| Published Tableau dashboard | 13,020 | 9 | Simple Stress Index `(demand / peak) × 100` | Live on Tableau Public |
| Current notebook export | 12,735 | 18 | Multi-signal Grid Stress Score (percentile-normalized) | Analytical enhancement |

Both versions use the same California-only scope, the same data cleaning steps, and the same review-priority classification thresholds (High ≥ 90, Medium ≥ 75). The difference is the scoring methodology and the set of records included in the dashboard export (all California records vs. complete-signal-only records).

The **README describes the published dashboard metrics** (13,020 hours, 75 High Review Priority, average Stress Index 48.62, peak demand 35,596 MW). The current notebook code represents the analytical evolution of the scoring approach.

The published Tableau dashboard remains the primary portfolio deliverable.

**Version-control note:** These generated CSV files are excluded from Git by `.gitignore`. They should be regenerated locally by running the notebook from top to bottom rather than committed to the repository.

## Step 19: Analytical Workflow Summary

This notebook implements a structured and reproducible workflow for analyzing California electricity grid conditions using public EIA-930 balancing authority data. The workflow moves from raw data ingestion to cleaned California-only records, exploratory analysis, multi-signal stress scoring, review-priority classification, and dashboard-ready exports.

The main workflow steps are:

1. **Load and inspect the raw EIA-930 dataset**  
   The notebook begins by loading the source file and reviewing its structure, column names, row count, and basic data types before transformation.

2. **Standardize and clean key fields**  
   Important columns are renamed for readability. Essential records are retained, duplicates are removed, timestamps are converted to Pacific Time, and core electricity measures are converted to numeric values.

3. **Validate analytical scope**  
   An initial full-dataset view revealed that the source file included balancing authorities from across the United States. The notebook explicitly investigates this issue before filtering to California-only records.

4. **Filter to California balancing authorities**  
   The analysis is narrowed to five California balancing authorities: BANC, CISO, IID, LDWP, and TIDC. This creates the main California working dataset.

5. **Explore California grid demand and supply patterns**  
   The notebook develops combined charts, panel charts, heatmaps, distribution plots, and weekday/weekend comparisons to study demand, forecast accuracy, generation balance, interchange behavior, hour-of-day patterns, and authority-specific differences.

6. **Engineer a multi-signal grid stress score**  
   The stress score combines four within-authority normalized components: demand pressure, forecast error pressure, generation-gap pressure, and import pressure. This design allows smaller authorities to be compared fairly with CISO.

7. **Classify review priority**  
   The continuous grid stress score is converted into Low, Medium, and High Review Priority categories using named threshold constants. These categories support dashboard filtering and review queues.

8. **Summarize key findings**  
   The notebook calculates core findings directly from the data, including scored record counts, review-priority shares, peak demand, average grid stress score, and the authority with the most High Review Priority periods.

9. **Export core analysis outputs**  
   The complete scored California stress dataset and the top review periods table are saved as reproducible project outputs.

10. **Build the dashboard data model**  
    Four purpose-built CSV files are exported for Tableau and the planned Power BI implementation: hourly detail, monthly summary, hourly risk summary, and KPI summary.

Every generated output file is excluded from version control by `.gitignore`. A reviewer with access to the raw EIA-930 data can reproduce the workflow by running this notebook from top to bottom.

## Step 20: Explore the Interactive Result

This notebook documents the analytical workflow behind the California Grid Analysis project, including data preparation, California-only scope validation, exploratory analysis, grid stress feature engineering, review-priority classification, and dashboard-ready export creation.

The exported dashboard data model is designed for Tableau and the planned Power BI implementation. The dashboard layer extends the notebook results into an interactive decision-support view focused on grid reliability, operational review, and risk-informed interpretation.

The dashboard is intended to help users explore:

- grid stress score trends over time,
- review-priority composition,
- high-priority risk concentration by balancing authority,
- hour-of-day stress patterns,
- top review periods,
- and KPI summaries by California balancing authority.

**Interactive Tableau Dashboard:**  
[Explore the California Grid Stress Monitoring Dashboard on Tableau Public](https://public.tableau.com/app/profile/sileshi.hirpa1285/viz/CaliforniaGridStressMonitoringDashboard/ExecutiveOverview)

## Step 21: Limitations and Caveats

This project uses public EIA-930 balancing authority data to build a portfolio-level California grid analysis workflow. The results are useful for analytical review, dashboard development, and decision-support demonstration, but they should be interpreted within the limits of the dataset and methodology.

### Dataset Window

The analysis covers the available January-April 2026 records in the working dataset. All summary statistics, grid stress scores, review-priority labels, and dashboard exports reflect this specific dataset window only.

### Public Data Only

This project uses public EIA-930 balancing authority data. It does not represent any utility’s internal operational systems, proprietary data pipelines, enterprise risk models, or reliability procedures.

### Grid Stress Score Is a Custom Analytical Indicator

The grid stress score is a purpose-built review indicator for this project. It combines demand pressure, forecast error pressure, generation-gap pressure, and import pressure using within-authority percentile normalization. It is not a formal operational risk score and does not replicate any utility or grid operator methodology.

### Relative, Not Absolute, Risk

The score is relative to the observed behavior of each balancing authority within this dataset window. A high score means a record is high relative to that authority’s own observed conditions, not necessarily high compared with an all-time historical record or formal reliability threshold.

### Missing Forecast Records

Records missing demand forecast values cannot receive a complete four-component stress score unless forecast values are imputed or the scoring formula is modified. This notebook uses complete stress-score records for dashboard export to keep the methodology transparent and consistent.

### No Real-Time Data Connection

The dataset is static. This pipeline does not automatically refresh as new EIA data becomes available.

### No Consequence or Impact Modeling

The project identifies periods with elevated grid stress signals. It does not model the probability, cause, or consequence of reliability events.

### Dashboard Implementation Status

The notebook exports a dashboard-ready data model for Tableau and the planned Power BI implementation. The interactive dashboard link should be added after the Tableau Public dashboard is built and published.

## Step 22: Conclusion and Portfolio Summary

### What This Notebook Demonstrates

This notebook implements an end-to-end analytical workflow for California electricity grid operations data. Starting from public EIA-930 balancing authority records, the project moves from raw data ingestion to cleaned California-only records, exploratory analysis, engineered stress scoring, review-priority classification, and dashboard-ready exports.

The workflow demonstrates the ability to:

1. load and inspect a raw public energy dataset,
2. clean and standardize key fields for analysis,
3. validate analytical scope before interpreting visualizations,
4. filter the dataset to California balancing authorities,
5. build explanatory visualizations for demand, forecast error, generation balance, interchange, and time-based patterns,
6. engineer a multi-signal grid stress score,
7. classify review priority for dashboard filtering,
8. summarize key findings directly from the data,
9. and export a structured dashboard data model for Tableau and Power BI.

### Why Scope Validation Was Important

One of the most important analytical decisions in the notebook was the scope validation step. An early full-dataset visualization ran without error, but it produced a misleading view because the source file included balancing authorities from across the United States.

This reinforced an important data-analysis principle: a chart can be technically correct and still answer the wrong question. By identifying the scope issue and filtering to California balancing authorities before continuing, the notebook keeps the analysis aligned with the project goal.

### Grid Stress Score as a Relative Review Indicator

The grid stress score is designed as a relative, multi-signal review indicator. It does not rank hours by raw demand alone. Instead, it combines four normalized components:

- demand pressure,
- forecast error pressure,
- generation-demand gap pressure,
- and import pressure.

Each component is normalized within its own balancing authority. This design allows smaller authorities such as IID and TIDC to appear in the review queue when they experience unusual pressure relative to their own operating patterns. Without this normalization, CISO would dominate most rankings because it has the largest absolute demand.

### Connection to the Dashboard Layer

The Phase 4 dashboard exports create four purpose-built CSV files for Tableau and the planned Power BI implementation:

- the hourly detail file supports time-series views, filters, review queues, and record-level drilldowns,
- the monthly summary file supports month-level trend views,
- the hourly risk summary supports hour-of-day stress profiles,
- and the KPI summary file supports per-authority dashboard scorecards.

The donut chart for review-priority composition and the horizontal bar chart for high-priority concentration by authority are intended as executive-summary dashboard visuals. They provide a quick view of overall review-priority distribution and show where High Review Priority periods are concentrated.

### Future Enhancements

Future versions of this project could improve the analysis by adding:

- a full-year or multi-year dataset for stronger seasonality analysis,
- rolling baseline features instead of fixed dataset-window percentile ranks,
- forecast-error decomposition by authority, hour, and day type,
- statistical anomaly detection using rolling z-scores or rolling percentiles,
- geographic mapping by balancing authority service territory,
- and a Power BI dashboard implementation using the same exported data model.

### Final Portfolio Statement

This project demonstrates a complete data science and business analytics workflow: data preparation, quality validation, exploratory analysis, feature engineering, prioritization logic, and dashboard data modeling.

The main value of the project is not only the final grid stress score, but the disciplined workflow behind it. The notebook shows how raw public data can be transformed into a structured analytical product that supports review, communication, and dashboard-based decision-making.

This project uses public EIA-930 data and should be interpreted as a portfolio-level analytical demonstration, not as an operational reliability system.